# Bài 26 · Từ mô hình ngôn ngữ đến trợ lý: SFT, học tăng cường với bộ kiểm tra, phần thưởng bị lách và DPO

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/26-hau-huan-luyen.ipynb)

Notebook đi kèm [Bài 26 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/hau-huan-luyen/) (reel và demo tương tác nằm ở trang bài học).
Sau tiền huấn luyện, mô hình chỉ biết viết tiếp văn bản. Hậu huấn luyện dạy nó trả lời, và dạy bằng phản hồi chứ không
chỉ bằng ví dụ mẫu. Ta bắt đầu với GPT-2 small thật (kết quả tính sẵn), rồi hậu huấn luyện mô hình đọc số của Bài 16.
Ở bài toán đó một chương trình chấm được mọi câu trả lời, nên ta đo chính xác những điều mà với chatbot thật chỉ đo
được gần đúng: phần thưởng, độ lệch KL, phần thưởng bị lách.

**Bạn sẽ làm:**
1. Xem GPT-2 small trả lời (hay không trả lời) một câu hỏi, và vì sao hai ví dụ hỏi–đáp thay đổi hẳn kết quả.
2. Viết hai bộ chấm cho bài đọc số, một chính xác và một cẩu thả, rồi chạy ba mô hình đã hậu huấn luyện của demo.
3. Tự viết học tăng cường kiểu GRPO, đơn giản hóa (REINFORCE với đường cơ sở trong nhóm và phạt KL), kiểm tra
   gradient, rồi chạy 100 bước thật.
4. Thấy phần thưởng bị lách (định luật Goodhart), tính lời giải đóng $\pi^* \propto \pi_{\text{ref}}\,e^{r/\beta}$, và
   học DPO trên một bảng nhỏ.
5. Đo tính toán lúc trả lời: bỏ phiếu đa số, best-of-N và độ lệch KL chính xác của nó.

**Cần biết trước:** [Bài 22 · Tiền huấn luyện & tinh chỉnh](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/tien-huan-luyen/),
[Bài 16 · Cơ chế chú ý](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/attention/) (mô hình đọc số mà ta hậu huấn
luyện) và độ phân kỳ KL của [Bài 19](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/autoencoder-vae/).
**Thời gian chạy:** dưới 1 phút trên CPU, kể cả tải bốn tệp kết quả (~0,6 MB) lần đầu.

Mọi con số trong notebook đều do chính các ô bên dưới tính ra (hoặc đọc từ tệp kết quả mà demo dùng), và trùng với
số trong demo.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn). Hàm `vn()` in số theo kiểu Việt Nam: dấu phẩy thập phân,
dấu chấm hàng nghìn. Python vẫn *gõ* số thập phân bằng dấu chấm (`0.1`), chỉ phần in ra là đổi.

In [ ]:
import base64
import hashlib
import json
import math
import urllib.request
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm một mã hóa thứ hai (hình dạng, nét gạch, nhãn)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38 (không in “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


def vn_tick(v, _pos=None):
    """Nhãn vạch trên trục: 0,5 · −2 · 60.000 (bỏ số 0 thừa ở cuối)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def log_tick(v, _pos=None):
    """Nhãn vạch trục log: 0,5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return vn_tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{vn_tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def vn_axes(fig):
    """Đổi nhãn mọi trục của hình (cả thanh màu) sang kiểu Việt Nam; gọi ngay trước plt.show().
    Trục log lấy vạch 1–2–5 nếu ngắn, hoặc các lũy thừa của 10 (cách đều, tối đa ~9 vạch) nếu dài."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(vn_tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


def prob(v):
    """Một xác suất: phần trăm từ 0,01%, nhỏ hơn thì viết kiểu 4,5·10⁻⁵."""
    return pct(v, 2) if v >= 1e-4 else sci(v)


def sci(v):
    """Số dương nhỏ kiểu 3,1·10⁻⁹."""
    e = math.floor(math.log10(v))
    return f"{vn(v / 10**e, 1)}·10{str(e).translate(SUP)}"


def span(vals, f=pct):
    """Khoảng thấp nhất–cao nhất giữa các seed."""
    lo, hi = f(min(vals)), f(max(vals))
    return lo if lo == hi else f"{lo}–{hi}"


print(vn(53259, 0), vn(0.0047, 4), pct(0.6735), prob(4.5e-5))

## 2 · Dữ liệu: bốn tệp kết quả của demo

GPT-2 small (124 triệu tham số) quá lớn cho notebook này, và việc huấn luyện các mô hình đọc số tốn hàng chục phút.
Ô dưới tải đúng bốn tệp mà demo dùng và kiểm tra MD5:
- `posttrain-base.json`: những gì GPT-2 small thật viết tiếp sau 9 câu lệnh (tính bằng mô hình thật của Bài 17);
- `posttrain-weights.json`: trọng số float16 của ba mô hình đọc số (seed 0): sau SFT, sau học tăng cường với bộ
  kiểm tra chính xác, và sau học tăng cường với bộ chấm cẩu thả;
- `posttrain-rl.json`: kết quả kiểm tra của mọi lần chạy (3 seed mỗi cấu hình), dùng ở mục 7–11;
- `posttrain-cands.json`: 12 câu hỏi của thẻ thứ hai trong demo, với các câu trả lời hay gặp và xác suất của chúng.

In [ ]:
# Kết quả của các lần chạy dài (cùng tệp với demo trên web), tải từ thư mục data/ của repo notebook công khai.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


base = fetch_json("posttrain-base.json", "fede87d4b05037589e51fe2d63c57351")  # GPT-2 small thật
shipped = fetch_json("posttrain-weights.json", "65903d4b01563135629bfa515122ab57")  # ba mô hình của demo
runs = fetch_json("posttrain-rl.json", "abfd281bb58763ae7f139a316f10fe7d")  # mọi lần chạy SFT / học tăng cường
cands = fetch_json("posttrain-cands.json", "d484589eac25dff1125ea1005706c827")  # 12 câu hỏi của thẻ 2
names = ("posttrain-base.json", "posttrain-weights.json", "posttrain-rl.json", "posttrain-cands.json")
size = sum((Path("data") / n).stat().st_size for n in names)
print(f"{len(names)} tệp, {vn(size / 1e6, 1)} MB · mô hình:", list(shipped["models"]))

## 3 · Mô hình nền viết tiếp, không trả lời

GPT-2 small chỉ được tiền huấn luyện để đoán token tiếp theo (Bài 17). Tệp `posttrain-base.json` ghi lại những gì nó
thật sự làm với 9 câu lệnh, mỗi câu đứng ngay sau token `<|endoftext|>`:
- xác suất của token đầu tiên ngay sau câu lệnh, và 5 token có xác suất cao nhất;
- 41 đoạn viết tiếp, mỗi đoạn 40 token: 1 đoạn tham lam, 20 đoạn với nhiệt độ $T = 0{,}7$ và 20 đoạn với $T = 1$,
  top-k 40 (cách lấy mẫu của Bài 17, 20 seed).

Câu hỏi mẫu là “What is the capital of France?”. Ta quan tâm token “ Paris” (có dấu cách ở đầu, đúng một token
trong bảng 50.257 token của GPT-2).

### 🤔 Dự đoán trước
Ngay sau câu hỏi trần “What is the capital of France?”, GPT-2 small đặt bao nhiêu xác suất vào token “ Paris”:
trên 50%, khoảng một phần mười, hay dưới 1%? Còn khi cùng câu hỏi đứng sau hai ví dụ hỏi–đáp (Ý → Rome, Nhật → Tokyo)?

In [ ]:
P = {p["id"]: p for p in base["prompts"]}
LABEL = {
    "q_en": "câu hỏi trần",
    "qa_en": "Q: … A:",
    "chat_en": "User: … Assistant:",
    "few_en": "2 ví dụ hỏi–đáp rồi câu hỏi",
    "stmt_en": "“The capital of France is”",
    "q_vi": "câu hỏi tiếng Việt",
}
paris = {}
for pid, label in LABEL.items():
    p = P[pid]
    paris[pid] = sum("Paris" in c["segs"][0] for c in p["cont"])  # đoạn viết tiếp (trước <|endoftext|>) có nhắc Paris
    print(
        f"{label:>28}: p(“ Paris”) = {prob(p['p']):>9}, hạng {p['rank']:>5}"
        f" · {paris[pid]}/{len(p['cont'])} đoạn viết tiếp có “Paris”"
    )
print(f"vocab: {vn(base['setup']['vocab'], 0)} token")

Với câu hỏi trần, “ Paris” chỉ có xác suất 0,47%, đứng thứ 17. Đặt cùng câu hỏi sau hai ví dụ hỏi–đáp thì nó lên
67,35% và đứng đầu. Mô hình **có** biết Paris; cái nó thiếu là *thói quen trả lời*. Ô dưới in 5 token đầu có xác suất
cao nhất trong hai trường hợp, và vài đoạn viết tiếp thật.

In [ ]:
def show(text, width=110):
    """Một đoạn viết tiếp trên một dòng: ⏎ thay cho xuống dòng."""
    s = text.replace("\n", " ⏎ ").strip()
    return s if len(s) <= width else s[: width - 1] + "…"


for pid in ("q_en", "few_en", "stmt_en"):
    top = " · ".join(f"{t!r} {pct(v)}" for t, v in P[pid]["top5"])
    print(f"{LABEL[pid]}: {top}")
print()
for pid, setting, seed in (("q_en", "greedy", 0), ("q_en", "T0.7", 0), ("q_en", "T0.7", 1), ("few_en", "T0.7", 0)):
    c = next(c for c in P[pid]["cont"] if c["setting"] == setting and c["seed"] == seed)
    print(f"[{LABEL[pid]}, {setting}, seed {seed}]  {show(c['segs'][0])}")
new_q = sum(c["newQ"] for c in P["few_en"]["cont"])
print(f"\nsau 2 ví dụ: {new_q}/{len(P['few_en']['cont'])} đoạn tự viết thêm một dòng “Q:” mới")

Với câu hỏi trần, token có xác suất cao nhất là xuống dòng (49,03%): GPT-2 coi câu hỏi là phần đầu của một trang web
và viết tiếp trang đó. Bản tham lam lặp mãi “The capital of France is Paris.”; một mẫu ở $T = 0{,}7$ nói thủ đô là
Lausanne (một thành phố của Thụy Sĩ). Sau hai ví dụ hỏi–đáp, nó trả lời Paris nhưng không dừng lại: nó viết tiếp *cả
trang hỏi–đáp*, với câu hỏi mới do nó tự đặt.

Hình dưới vẽ xác suất của “ Paris” (trục log) cho các câu lệnh tiếng Anh: thanh xanh dương là câu lệnh không có ví dụ,
thanh cam có gạch chéo là câu lệnh có hai ví dụ.

In [ ]:
order = ["stmt_en", "q_en", "chat_en", "qa_en", "few_en"]
vals = [P[k]["p"] for k in order]
fig, ax = plt.subplots(figsize=(8, 2.8))
bars = ax.barh(range(len(order)), vals, color=[ORANGE if k == "few_en" else BLUE for k in order])
bars[-1].set_hatch("//")
for i, v in enumerate(vals):
    ax.text(v * 1.15, i, prob(v), va="center", fontsize=9)
ax.set_yticks(range(len(order)), labels=[LABEL[k] for k in order])
ax.set(xscale="log", xlim=(1e-3, 3), xlabel="p(“ Paris”) ngay sau câu lệnh (thang log)")
ax.set_title("GPT-2 small biết Paris, nhưng chỉ nói ra khi câu lệnh giống một trang hỏi–đáp")
vn_axes(fig)
plt.show()

Kể cả câu “The capital of France is” cũng chỉ cho “ Paris” 0,16%: mô hình hay viết “now”, “the”, “a”…, vì trên web câu
ấy thường chưa kết thúc ở đó. Ta thấy hai điều:
- **Kiến thức có sẵn từ tiền huấn luyện.** Hai ví dụ không dạy GPT-2 điều gì mới (không có gradient nào); chúng chỉ
  chọn ra *loại văn bản* mà mô hình viết tiếp.
- **Hậu huấn luyện dạy hành vi.** Bước đầu là tinh chỉnh có giám sát (SFT): vẫn là cross-entropy của Bài 22, nhưng trên
  các cặp câu lệnh–câu trả lời mẫu, để mô hình trả lời rồi dừng. InstructGPT dùng khoảng 13 nghìn câu lệnh có câu trả
  lời do người viết; LIMA (Zhou và cộng sự, 2023) cho thấy 1.000 ví dụ chọn kỹ đã đủ để dạy định dạng.

Đo chất lượng câu trả lời của một chatbot thật là việc khó: cần người chấm, hoặc một mô hình chấm. Phần còn lại của
notebook dùng một bài toán mà **một chương trình chấm được chính xác**, nên ta đo được mọi thứ.

## 4 · Bài toán đọc số và hai bộ chấm

Mô hình của Bài 16 đọc cách đọc tiếng Việt của một số có 1–12 chữ số (“năm nghìn hai trăm”) rồi viết các chữ số
(5200), kết thúc bằng ký hiệu ⟨hết⟩. Ô dưới chép lại bộ đọc số và bộ lấy mẫu của Bài 16 (giống hệt
`reel/attention/numvi.py`): mỗi chữ số bằng 0 với xác suất 0,25, mỗi nhóm ba chữ số bên dưới nhóm đầu bằng 000 với xác
suất 0,2.

In [ ]:
D = ["không", "một", "hai", "ba", "bốn", "năm", "sáu", "bảy", "tám", "chín"]
SCALE = ["", "nghìn", "triệu", "tỷ"]


def tens_units(b, c):
    """Hàng chục b và hàng đơn vị c của một nhóm ba chữ số."""
    if b == 0:
        return ["linh", D[c]] if c else []
    if b == 1:
        return ["mười"] + (["lăm" if c == 5 else D[c]] if c else [])
    return [D[b], "mươi"] + ([{1: "mốt", 4: "tư", 5: "lăm"}.get(c, D[c])] if c else [])


def group(g, lead):
    a, b, c = g // 100, g // 10 % 10, g % 10
    if lead and a == 0:  # nhóm đầu tiên không đọc số 0 ở đầu
        return [D[c]] if b == 0 else tens_units(b, c)
    return [D[a], "trăm"] + tens_units(b, c)


def read(n):
    """Số nguyên n ≥ 0 → danh sách các từ."""
    if n == 0:
        return ["không"]
    groups = []
    while n:
        groups.append(n % 1000)
        n //= 1000
    words = []
    for k in reversed(range(len(groups))):
        lead = k == len(groups) - 1
        if groups[k] == 0 and not lead:  # bỏ nhóm 000 ở giữa
            continue
        words += group(groups[k], lead)
        if SCALE[k]:
            words.append(SCALE[k])
    return words


def sample(rng, d, n, p0=0.25, pg=0.2):
    """n số có đúng d chữ số, mỗi hàng là các chữ số."""
    digs = np.where(rng.random((n, d)) < p0, 0, rng.integers(1, 10, (n, d)))
    digs[:, 0] = rng.integers(1, 10, n)  # chữ số đầu khác 0
    for g0 in range(d - 3, 0, -3):  # các nhóm ba chữ số tính từ bên phải, trừ nhóm đầu
        digs[rng.random(n) < pg, g0 : g0 + 3] = 0
    return digs


for n in (5200, 150, 30500, 554000):
    print(f"{vn(n, 0):>8} → {' '.join(read(n))}")
print(f"bộ lấy mẫu: mỗi chữ số là 0 với xác suất {vn(0.25)}, mỗi nhóm 000 với xác suất {vn(0.2, 1)}")
assert " ".join(read(7090400)) == "bảy triệu không trăm chín mươi nghìn bốn trăm"

Học tăng cường cần một **phần thưởng** cho mỗi câu trả lời. Ở đây có một bộ kiểm tra chính xác mà **không cần đáp
án**: nó đọc to các chữ số của câu trả lời bằng `read()` rồi so với các từ của câu hỏi. Câu trả lời phải kết thúc
bằng ⟨hết⟩ trong 13 token, và không được bắt đầu bằng 0 (trừ chính số 0).

Bộ thứ hai là một bộ chấm **cẩu thả**, kiểu thước đo gần đúng hay gặp ngoài đời: nó chỉ so các chữ số khác 0 với các
từ chỉ chữ số khác 0 trong câu hỏi, theo đúng thứ tự, và bỏ qua mọi chữ số 0. Cả hai đều giống `reel/posttrain/rl.py`.

Token đầu ra: 0–9 là chữ số, 10 là ⟨hết⟩; bộ giải mã bắt đầu từ ký hiệu 11.

In [ ]:
EOS, BOS, TMAX, N_OUT = 10, 11, 13, 11
NZ = {"một": 1, "mốt": 1, "hai": 2, "ba": 3, "bốn": 4, "tư": 4, "năm": 5, "lăm": 5, "sáu": 6, "bảy": 7, "tám": 8}
NZ |= {"chín": 9, "mười": 1}  # mỗi từ chỉ một chữ số khác 0 (“mười” là chữ số 1 của hàng chục)


def tokens(digits):
    """Chuỗi chữ số → các token của câu trả lời đúng, kết thúc bằng ⟨hết⟩."""
    return [int(c) for c in digits] + [EOS]


def answer_len(out):
    """Số token tính đến ⟨hết⟩ đầu tiên (13 nếu không có)."""
    has = out == EOS
    return np.where(has.any(1), has.argmax(1) + 1, TMAX)


def verify(rows, out):
    """Bộ kiểm tra chính xác: 1 nếu các chữ số của câu trả lời, đọc to lên, cho đúng các từ của câu hỏi."""
    L, r = answer_len(out), np.zeros(len(out))
    for b, row in enumerate(rows):
        dg = out[b, : L[b] - 1]
        if out[b, L[b] - 1] != EOS or len(dg) == 0 or (dg[0] == 0 and len(dg) > 1):
            continue
        r[b] = float(read(int("".join(map(str, dg)))) == read(int(row)))
    return r


def proxy(rows, out):
    """Bộ chấm cẩu thả: các chữ số khác 0, theo thứ tự, khớp các từ chỉ chữ số khác 0; chữ số đầu khác 0."""
    L, r = answer_len(out), np.zeros(len(out))
    for b, row in enumerate(rows):
        dg = out[b, : L[b] - 1]
        if out[b, L[b] - 1] != EOS or len(dg) == 0 or dg[0] == 0:
            continue
        r[b] = float([v for v in dg if v != 0] == [NZ[w] for w in read(int(row)) if w in NZ])
    return r


def pad(answers):
    """Các câu trả lời (chuỗi chữ số) → mảng token (n, 13), đệm bằng ⟨hết⟩."""
    return np.array([tokens(a) + [EOS] * (TMAX - len(a) - 1) for a in answers])


tries = ["150", "15000", "1500", "105", "15", "1050"]
print("câu hỏi: “một trăm năm mươi”")
for a, ok, px in zip(tries, verify(["150"] * 6, pad(tries)), proxy(["150"] * 6, pad(tries))):
    print(f"  {a:>6}: bộ kiểm tra {int(ok)} · bộ chấm cẩu thả {int(px)}")
assert verify(["150"], pad(["15000"]))[0] == 0 and proxy(["150"], pad(["15000"]))[0] == 1

Bộ chấm cẩu thả nhận cả 15000, 1500, 105, 15 và 1050 cho “một trăm năm mươi”. Nó nhận bao nhiêu câu trả lời cho một
câu hỏi? Các chữ số khác 0 có thứ tự cố định, chữ số đầu là chữ số khác 0 đầu tiên, còn các chữ số 0 chen vào đâu cũng
được. Ô dưới liệt kê mọi câu trả lời nó nhận, với tối đa 12 chữ số.

In [ ]:
def proxy_set(row):
    """Mọi câu trả lời (≤ 12 chữ số) mà bộ chấm cẩu thả nhận cho câu hỏi của số row."""
    want = [NZ[w] for w in read(int(row)) if w in NZ]
    out = []
    for length in range(len(want), 13):
        for pos in combinations(range(1, length), len(want) - 1):  # vị trí của các chữ số khác 0 sau chữ số đầu
            dg = [0] * length
            dg[0] = want[0]
            for k, j in enumerate(pos):
                dg[j] = want[k + 1]
            out.append("".join(map(str, dg)))
    return out


for row in ("150", "5200", "7090400"):
    acc = proxy_set(row)
    m = sum(w in NZ for w in read(int(row)))
    print(
        f"{row:>8}: {m} chữ số khác 0 → bộ chấm cẩu thả nhận {vn(len(acc), 0)} câu trả lời (C(12, {m}) = "
        f"{vn(math.comb(12, m), 0)}); chỉ 1 câu đúng"
    )
    assert len(acc) == math.comb(12, m) and row in acc
    assert proxy([row] * len(acc), pad(acc)).all() and verify([row] * len(acc), pad(acc)).sum() == 1

## 5 · Ba mô hình đọc số đã hậu huấn luyện

Mô hình là bộ mã hóa–giải mã GRU có attention cộng tính của Bài 16 (H = 64). Demo dùng ba bản của nó, cùng seed 0:
- **SFT-240:** học có giám sát (cross-entropy) trên chỉ 240 số có nhãn, 20 số cho mỗi độ dài; đây là $\pi_{\text{ref}}$;
- **RL, bộ kiểm tra:** từ SFT-240, 1.000 bước học tăng cường với phần thưởng của `verify()`, $\beta = 0$;
- **RL, bộ chấm cẩu thả:** như trên, nhưng phần thưởng là `proxy()`.

Học tăng cường ở đây không dùng nhãn nào: mỗi bước rút 16 số mới, mô hình tự đọc, bộ chấm cho điểm. Ô dưới mở trọng
số float16 (như demo) và đếm tham số.

In [ ]:
SRC = ["<pad>", *D, "mốt", "tư", "lăm", "mười", "mươi", "trăm", "linh", "nghìn", "triệu", "tỷ"]  # id 0 là phần đệm
W2I = {w: i for i, w in enumerate(SRC)}
assert shipped["src"] == SRC


def unpack(model):
    """float16 mã hóa base64 → mảng float64, như demo."""
    return {
        k: np.frombuffer(base64.b64decode(v["f16"]), "<f2").astype(np.float64).reshape(v["shape"])
        for k, v in model.items()
    }


MODELS = {name: unpack(m) for name, m in shipped["models"].items()}
NAME = {"sft": "SFT-240", "rl": "RL, bộ kiểm tra", "proxy": "RL, bộ chấm cẩu thả"}
n_params = sum(v.size for v in MODELS["sft"].values())
print({k: v.shape for k, v in MODELS["sft"].items()})
print(f"{vn(n_params, 0)} tham số mỗi mô hình")
assert n_params == runs["setup"]["params"] == 53259


def questions(rows):
    """Các số (chuỗi chữ số) → id từ của câu hỏi X (đệm 0) và mặt nạ M (1 = có từ)."""
    ids = [[W2I[w] for w in read(int(r))] for r in rows]
    X = np.zeros((len(ids), max(map(len, ids))), int)
    for i, s in enumerate(ids):
        X[i, : len(s)] = s
    return X, (X > 0).astype(float)


def qset(base, n):
    """n số cho mỗi độ dài 1–12 từ default_rng(base + d), như Bài 16: kiểm định 1000 + d, kiểm tra 2000 + d."""
    rows = ["".join(map(str, r)) for d in range(1, 13) for r in sample(np.random.default_rng(base + d), d, n)]
    return rows, *questions(rows)


TEST, VAL = qset(2000, 100), qset(1000, 50)
digits = np.array([len(r) for r in TEST[0]])
print(f"tập kiểm tra: {vn(len(TEST[0]), 0)} số · tập kiểm định: {vn(len(VAL[0]), 0)} số")

Các phương trình là của Bài 16, theo quy ước hàng $z = xW + b$. Lần này hàm GRU trả thêm các giá trị trung gian
(`cache`), vì mục 8 cần lan truyền ngược qua chúng.

In [ ]:
def sig(v):
    return 0.5 * (1 + np.tanh(0.5 * v))  # = 1 / (1 + e^(−v)), không bao giờ tràn số


def gru(ax, h, U):
    """Một bước GRU; ax = xW + b của cả ba phần (cổng đặt lại r, cổng cập nhật z, giá trị đề cử n), nối theo cột."""
    H = h.shape[1]
    rz = sig(ax[:, : 2 * H] + h @ U[:, : 2 * H])
    r, z = rz[:, :H], rz[:, H:]
    n = np.tanh(ax[:, 2 * H :] + (r * h) @ U[:, 2 * H :])
    return z * h + (1 - z) * n, (h, r, z, r * h, n)


def encode(p, X, M):
    """Bộ mã hóa GRU: các trạng thái Hs (B, S, H), trạng thái cuối h_T và cache của từng bước."""
    h = np.zeros((X.shape[0], p["Ue"].shape[0]))
    Hs, caches = np.zeros((*X.shape, h.shape[1])), []
    for j in range(X.shape[1]):
        m = M[:, j : j + 1]
        hn, c = gru(p["Ex"][X[:, j]] + p["be"], h, p["Ue"])
        h = m * hn + (1 - m) * h  # phần đệm (M = 0) giữ nguyên h
        Hs[:, j] = h
        caches.append(c)
    return Hs, h, caches


def attend(p, s, Hs, keys, M):
    """Attention cộng tính: e_j = tanh(s W_a + h_j U_a) · v_a, softmax, ngữ cảnh c = Σ α_j h_j. keys = Hs @ U_a."""
    t = np.tanh(keys + (s @ p["Wa"])[:, None, :])
    e = np.where(M > 0, t @ p["va"], -np.inf)
    a = np.exp(e - e.max(1, keepdims=True))
    a /= a.sum(1, keepdims=True)
    return np.einsum("bs,bsh->bh", a, Hs), (a, t)


def logits(p, s, y, Hs, keys, M):
    """Một bước giải mã sau token y: trạng thái mới, logits của 11 đầu ra, và cache."""
    c, acache = attend(p, s, Hs, keys, M)
    sn, gcache = gru(p["Ey"][y] + c @ p["Wc"] + p["bd"], s, p["Ud"])
    return sn, sn @ p["Wo"] + c @ p["Wco"] + p["bo"], (s, c, acache, gcache, sn)


def generate(p, X, M, U=None, rep=1):
    """Viết câu trả lời, tối đa 13 token. U = None: tham lam (đầu ra có logit lớn nhất). Ngược lại U (13, B·rep) là các
    số đều trong [0, 1): ở bước i, hàng b chọn đầu ra đầu tiên mà xác suất tích lũy vượt U[i, b], như demo và reel.
    rep: số câu trả lời cho mỗi câu hỏi (bộ mã hóa chỉ chạy một lần cho mỗi câu hỏi)."""
    Hs, h_T, _ = encode(p, X, M)
    keys = Hs @ p["Ua"]
    q = np.repeat(np.arange(len(X)), rep)  # câu hỏi của mỗi hàng
    out = np.full((len(q), TMAX), EOS)
    act, y, s = np.arange(len(q)), np.full(len(q), BOS), h_T[q]  # các hàng chưa viết ⟨hết⟩
    for i in range(TMAX):
        qa = q[act]
        s, z, _ = logits(p, s, y, Hs[qa], keys[qa], M[qa])
        if U is None:
            y = z.argmax(1)
        else:
            pr = np.exp(z - z.max(1, keepdims=True))
            pr /= pr.sum(1, keepdims=True)
            y = np.minimum((pr.cumsum(1) < U[i, act][:, None]).sum(1), N_OUT - 1)
        out[act, i] = y
        keep = y != EOS
        act, y, s = act[keep], y[keep], s[keep]
        if not len(act):
            break
    return out


def text(out):
    """Các token → chuỗi chữ số (“…” nếu không có ⟨hết⟩)."""
    L = answer_len(out)
    return ["".join(map(str, o[: n - 1])) if o[n - 1] == EOS else "".join(map(str, o)) + "…" for o, n in zip(out, L)]

Chạy cả ba mô hình (tham lam) trên 1.200 số kiểm tra, 100 cho mỗi độ dài, và chấm bằng cả hai bộ chấm. Ô cũng so với
kết quả seed 0 mà các lần chạy đã lưu, để chắc rằng bản numpy này chính là mô hình của demo.

### 🤔 Dự đoán trước
Bộ chấm cẩu thả nhận mọi câu trả lời đúng, cộng thêm nhiều câu sai. Mô hình học tăng cường theo nó, so với SFT-240
ban đầu, sẽ đọc đúng **nhiều hơn**, **bằng**, hay **ít hơn**? Và so với mô hình học theo bộ kiểm tra chính xác?

In [ ]:
rows, X, M = TEST
STORED = {  # kết quả kiểm tra đã lưu của đúng ba mô hình này (seed 0)
    "sft": runs["sft"]["240"]["seeds"][0]["test"],
    "rl": runs["rl"]["b0.0"]["seeds"][0]["test"],
    "proxy": runs["rl"]["proxy0.0"]["seeds"][0]["test"],
}
greedy = {}
for name, p in MODELS.items():
    out = generate(p, X, M)
    greedy[name] = {"ok": verify(rows, out), "px": proxy(rows, out), "text": text(out)}
    g = greedy[name]
    print(f"{NAME[name]:>20}: đọc đúng {pct(g['ok'].mean())} · bộ chấm cẩu thả cho {pct(g['px'].mean())}")
    for key, k in (("ok", "greedy"), ("px", "greedyProxy")):
        assert g[key].sum() == round(len(rows) * STORED[name][k]), "khác với demo: hãy kiểm tra lại mô hình"

Từ 240 ví dụ có nhãn, SFT đọc đúng 55,50% số thử (mục 6 cho thấy nó đọc đúng cả 240 số nó đã học).
Học tăng cường với bộ kiểm tra đưa con số này lên 78,17% mà không cần thêm nhãn nào. Học theo bộ chấm cẩu thả thì
điểm của bộ chấm lên 92,00%, nhưng tỉ lệ đọc đúng thật chỉ 59,67%: nhỉnh hơn SFT ở seed này, kém xa mô hình học theo
bộ kiểm tra, và mục 7 cho thấy ở 2 trong 3 seed nó còn tụt dưới cả SFT. Lý do: phần thưởng không phân biệt câu đúng
với các câu sai mà bộ chấm nhận, nên học tăng cường chỉ đẩy xác suất về phía *cả nhóm* đó, câu nào dễ viết thì được
lợi. Ô dưới in câu trả lời của ba mô hình cho vài câu hỏi, kèm nhãn: “đúng”, “cẩu thả” (chỉ bộ chấm cẩu thả nhận)
hoặc “sai”.

In [ ]:
hacked = (greedy["proxy"]["px"] == 1) & (greedy["proxy"]["ok"] == 0)
picks = [int(np.flatnonzero(hacked & (digits == d))[0]) for d in (3, 5, 8, 11)]
for i in picks:
    marks = []
    for name in MODELS:
        g = greedy[name]
        tag = "đúng" if g["ok"][i] else ("cẩu thả" if g["px"][i] else "sai")
        marks.append(f"{NAME[name]}: {g['text'][i]:>12} {tag:<7}")
    print(f"{rows[i]:>12} “{' '.join(read(int(rows[i])))}”")
    print("              " + " · ".join(marks))

Mô hình học theo bộ chấm cẩu thả đã học đúng chỗ hở của bộ chấm: nó thêm hoặc bớt chữ số 0. Ô dưới đếm, trong các câu
trả lời tham lam mà chỉ bộ chấm cẩu thả nhận, bao nhiêu câu **dài hơn** (thừa chữ số 0) và bao nhiêu câu **ngắn hơn**
(thiếu chữ số 0) so với đáp án, theo số chữ số của câu hỏi.

In [ ]:
for name in MODELS:
    g = greedy[name]
    fake = (g["px"] == 1) & (g["ok"] == 0)
    dlen = np.array([len(a) - len(r) for a, r in zip(g["text"], rows)])  # độ dài câu trả lời − độ dài đáp án
    print(
        f"{NAME[name]:>20}: {fake.sum():>3} câu chỉ bộ cẩu thả nhận · thừa số 0: {(fake & (dlen > 0)).sum():>3}"
        f" · thiếu số 0: {(fake & (dlen < 0)).sum():>3} · số 0 sai chỗ: {(fake & (dlen == 0)).sum():>2}"
    )
g = greedy["proxy"]
fake = (g["px"] == 1) & (g["ok"] == 0)
dlen = np.array([len(a) - len(r) for a, r in zip(g["text"], rows)])  # độ dài câu trả lời − độ dài đáp án
print("RL, bộ chấm cẩu thả, theo số chữ số d = 1…12 (trên 100 câu mỗi d):")
for label, m in (
    ("đúng thật", g["ok"] == 1),
    ("thừa số 0", fake & (dlen > 0)),
    ("thiếu số 0", fake & (dlen < 0)),
    ("số 0 sai chỗ", fake & (dlen == 0)),
):
    print(f"  {label:>12}:", [int(m[digits == d].sum()) for d in range(1, 13)])

Với số 3–6 chữ số, mô hình học theo bộ cẩu thả hay *thêm* một chữ số 0 (74508 → 745080). Từ 7 chữ số trở đi, nó hầu
như chỉ *bớt* chữ số 0 (48201456040 → 4820145604): đọc đúng một số 11 chữ số là việc khó, còn viết các chữ số khác 0
theo thứ tự rồi bỏ bớt vài chữ số 0 thì dễ hơn nhiều mà vẫn được thưởng. Mô hình không “gian lận” có chủ ý; nó chỉ tối đa đúng con số ta đưa cho nó.

## 6 · Xác suất chính xác của một câu trả lời

Một câu trả lời $y = (y_1, \dots, y_L)$ là một chuỗi token, và mô hình cho xác suất của cả chuỗi bằng tích các xác suất
từng bước: $\pi(y \mid x) = \prod_t \pi(y_t \mid x, y_{<t})$. Muốn tính nó, ta cho bộ giải mã đọc chính câu trả lời
đó (teacher forcing, như lúc SFT) và cộng $\log \pi(y_t \mid \dots)$ của các token đến ⟨hết⟩. Hàm `forward` dưới đây làm
việc đó và giữ lại cache cho lan truyền ngược ở mục 7.

In [ ]:
def softmax(z):
    a = np.exp(z - z.max(-1, keepdims=True))
    return a / a.sum(-1, keepdims=True)


def forward(p, X, M, Y):
    """Teacher forcing: xác suất (B, T, 11) của mọi token kế tiếp khi bộ giải mã đọc các token của Y, và cache."""
    Hs, h_T, ecache = encode(p, X, M)
    keys = Hs @ p["Ua"]
    Yin = np.concatenate([np.full((len(Y), 1), BOS), Y[:, :-1]], 1)  # đầu vào của bước i là token i − 1
    s, probs, dcache = h_T, [], []
    for i in range(Y.shape[1]):
        s, z, cache = logits(p, s, Yin[:, i], Hs, keys, M)
        probs.append(softmax(z))
        dcache.append(cache)
    return np.stack(probs, 1), (X, M, Yin, Hs, ecache, dcache)


def seq_logp(P, Y, L):
    """log π(y) = Σ_{t < L} log P[t, y_t] cho mỗi hàng, và mặt nạ các vị trí được tính."""
    mask = np.arange(Y.shape[1])[None] < L[:, None]
    lp = np.log(np.take_along_axis(P, Y[:, :, None], 2)[:, :, 0])
    return (lp * mask).sum(1), mask


def logprob(p, rows, answers):
    """log π(câu trả lời | câu hỏi) chính xác cho từng cặp (số trong câu hỏi, câu trả lời), mỗi câu là chuỗi chữ số."""
    X, M = questions(rows)
    Y = pad(answers)
    return seq_logp(forward(p, X, M, Y)[0], Y, np.array([len(a) + 1 for a in answers]))[0]

Lấy câu hỏi “năm nghìn hai trăm” (câu mặc định ở thẻ 2 của demo). Ô dưới cho SFT-240 viết 8 câu trả lời bằng cách
lấy mẫu ($T = 1$, seed 1), rồi tính xác suất chính xác của 4 câu trả lời mà nó hay viết nhất, và so với xác suất mà
tệp của demo đã lưu.

In [ ]:
q5200 = next(c for c in cands["questions"] if c["digits"] == "5200")
rng = np.random.default_rng(1)
X1, M1 = questions(["5200"])
samples8 = text(generate(MODELS["sft"], X1, M1, rng.random((TMAX, 8)), rep=8))
print("8 mẫu của SFT-240:", samples8, f"· {samples8.count('5200')}/8 mẫu đúng")
top = [c["y"] for c in q5200["cands"]][:4]
lp = logprob(MODELS["sft"], ["5200"] * len(top), top)
for a, v, c in zip(top, lp, q5200["cands"]):
    tag = "đúng" if verify(["5200"], pad([a]))[0] else ("cẩu thả" if proxy(["5200"], pad([a]))[0] else "sai")
    print(f"  {a:>5} ({tag:>7}): log π = {vn(v, 3):>7} · π = {pct(math.exp(v))}")
    assert abs(math.exp(v) - c["p"]["sft"]) < 1e-6 * max(1, c["p"]["sft"]) + 1e-9, "khác với demo"

SFT-240 viết “520” với xác suất 70,00% và đáp án đúng “5200” chỉ 29,28%: nó *biết* đáp án, nhưng câu sai lại được ưa
hơn. Bộ chấm cẩu thả nhận cả hai.

Bộ kiểm tra chỉ nhận đúng **một** câu trả lời cho mỗi câu hỏi (cách đọc số là duy nhất), nên xác suất của đáp án đúng
$\pi(\text{đúng} \mid x)$ cũng chính là **điểm kỳ vọng** của một câu trả lời lấy mẫu, tính chính xác, không cần lấy
mẫu. Ô dưới tính nó cho cả 1.200 câu thử và ba mô hình, rồi so với tệp kết quả.

In [ ]:
pc = {name: np.exp(logprob(p, rows, rows)) for name, p in MODELS.items()}  # đáp án đúng của số r là chính r
for name in MODELS:
    print(f"{NAME[name]:>20}: điểm kỳ vọng của một mẫu = trung bình π(đúng) = {pct(pc[name].mean())}")
    assert abs(pc[name].mean() - STORED[name]["sample"]) < 1e-4, "khác với demo"
ref_pc = np.array(runs["ref"]["pc"])
assert np.allclose(pc["sft"], ref_pc, rtol=1e-4, atol=1e-6), "khác với demo"
print(f"{(pc['sft'] < 1e-6).sum()} câu mà SFT-240 cho đáp án đúng xác suất dưới một phần triệu")

SFT chính là cực tiểu hóa $-\log \pi(y^{\text{mẫu}} \mid x)$ trên các cặp mẫu: cùng hàm `logprob` ở trên, với $y$ là câu
trả lời có nhãn. Ô dưới dựng lại 240 số mà SFT-240 đã học (20 cho mỗi độ dài, rút từ `default_rng(10_000)` như
`rl.sft` của reel), rồi đo cross-entropy trung bình và tỉ lệ đọc đúng trên chính chúng.

In [ ]:
rng = np.random.default_rng(10_000 + 0)  # seed 0
sft_rows = ["".join(map(str, r)) for d in range(1, 13) for r in sample(rng, d, 20)]
ce = -logprob(MODELS["sft"], sft_rows, sft_rows)
seen = verify(sft_rows, generate(MODELS["sft"], *questions(sft_rows)))
print(f"240 số đã học: cross-entropy trung bình {vn(ce.mean(), 3)} nat mỗi câu · đọc đúng {pct(seen.mean())}")
print(
    f"1.200 số thử:  cross-entropy trung bình {vn(-np.log(pc['sft']).mean(), 3)} nat mỗi câu · đọc đúng "
    f"{pct(greedy['sft']['ok'].mean())}"
)
assert seen.all()

SFT-240 thuộc lòng 240 số nó đã học nhưng chỉ đọc đúng khoảng một nửa số mới: 240 ví dụ là quá ít để học hết luật
đọc số (Bài 08 gọi đó là quá khớp). Thêm nhãn là một cách; cách khác, mà phần còn lại của bài dùng, là cho mô hình tự
luyện trên các câu hỏi *không có nhãn* và để bộ kiểm tra chấm.

Ở seed này 3/8 mẫu là “5200”; với chỉ 8 mẫu, con số này dao động nhiều quanh 29,28% (đổi seed để thấy).

## 7 · Học tăng cường kiểu GRPO, đơn giản hóa

### Mục tiêu và gradient
Gọi $\pi_\theta$ là mô hình đang học (chính sách) và $\pi_{\text{ref}}$ là bản SFT, giữ cố định. Mục tiêu của RLHF
(InstructGPT) và của bài này là

$$J(\theta) = \mathbb{E}_{y \sim \pi_\theta}\big[r(x, y)\big] - \beta\, D_{\mathrm{KL}}\big(\pi_\theta \,\|\, \pi_{\text{ref}}\big).$$

Không lấy đạo hàm được qua phép lấy mẫu, nhưng có đẳng thức $\nabla_\theta \mathbb{E}_{y \sim \pi_\theta}[f(y)] =
\mathbb{E}_{y \sim \pi_\theta}[f(y)\, \nabla_\theta \log \pi_\theta(y)]$ (REINFORCE, Williams 1992). Áp dụng cho cả hai
số hạng (số hạng thêm vào của KL có kỳ vọng 0, vì $\mathbb{E}[\nabla \log \pi_\theta] = 0$):

$$\nabla_\theta J = \mathbb{E}_{y \sim \pi_\theta}\big[R(y)\, \nabla_\theta \log \pi_\theta(y)\big], \qquad R(y) = r(y) - \beta \log \frac{\pi_\theta(y)}{\pi_{\text{ref}}(y)}.$$

Phạt KL chỉ là một khoản trừ vào phần thưởng của từng câu trả lời: câu nào mô hình đã ưa hơn hẳn $\pi_{\text{ref}}$ thì
bị trừ nhiều. InstructGPT trừ đúng khoản này ở từng token, với $\beta = 0{,}02$.

### Đường cơ sở trong nhóm
Với mỗi câu hỏi ta lấy $G = 8$ câu trả lời. Trừ vào $R_i$ một số $b$ không phụ thuộc $y_i$ không đổi kỳ vọng của
gradient nhưng giảm phương sai. Ta lấy trung bình của **7 câu trả lời còn lại** cho cùng câu hỏi (leave-one-out, như
RLOO của Ahmadian và cộng sự, 2024):

$$A_i = R_i - \frac{1}{G - 1} \sum_{j \neq i} R_j.$$

GRPO (Shao và cộng sự, 2024; DeepSeek-R1) cũng dùng nhóm thay cho mạng giá trị của PPO, nhưng trừ trung bình cả nhóm
rồi chia độ lệch chuẩn của nhóm, thêm phép cắt tỉ số $\pi_\theta / \pi_{\text{old}}$ của PPO, và đặt phạt KL thẳng vào
hàm mất mát thay vì trừ vào phần thưởng. Ở đây mỗi lô chỉ cập nhật một lần nên không cần cắt; vì vậy khóa học gọi nó
là **kiểu GRPO, đơn giản hóa**.

### Từ công thức đến code
Gradient trên là gradient của hàm mất mát thay thế (surrogate) $\mathcal{L} = -\frac{1}{N}\sum_i A_i \log \pi_\theta(y_i)$
khi coi $A_i$ là hằng số, với $N = 16 \cdot 8$ câu trả lời mỗi bước. Theo logits ở bước $t$, đạo hàm là
$\frac{A_i}{N}\,(p_t - \text{onehot}(y_t))$: giống hệt cross-entropy của SFT, chỉ nhân với $A_i$. $A_i > 0$ đẩy xác suất
của câu đó lên, $A_i < 0$ đẩy xuống. Ô dưới viết lan truyền ngược qua bộ giải mã có attention và qua bộ mã hóa GRU
(giống `rl.tf` của reel); ô sau đó kiểm tra nó bằng sai phân hữu hạn.

In [ ]:
def gru_back(dh_new, cache, U, dU):
    """Lan truyền ngược một bước GRU: gradient theo ax và theo trạng thái cũ h; cộng dồn gradient của U vào dU."""
    h, r, z, rh, n = cache
    H = h.shape[1]
    dz, dn, dh = dh_new * (h - n), dh_new * (1 - z), dh_new * z  # h_new = z·h + (1 − z)·n
    dan = dn * (1 - n**2)  # qua tanh
    drh = dan @ U[:, 2 * H :].T
    dh += drh * r
    darz = np.concatenate([drh * h * r * (1 - r), dz * z * (1 - z)], 1)  # qua hai sigmoid
    dU[:, : 2 * H] += h.T @ darz
    dU[:, 2 * H :] += rh.T @ dan
    dh += darz @ U[:, : 2 * H].T
    return np.concatenate([darz, dan], 1), dh


def backward_decoder(p, g, cache, dZ):
    """Bộ giải mã, từ bước cuối về bước đầu. Trả về gradient theo Hs, theo keys = Hs @ U_a, và theo h_T."""
    _, _, Yin, Hs, _, dcache = cache
    dHs, dkeys, ds = np.zeros_like(Hs), np.zeros((*Hs.shape[:2], p["Ua"].shape[1])), np.zeros_like(Hs[:, 0])
    for i in reversed(range(len(dcache))):
        s, c, (a, t), gcache, sn = dcache[i]
        dz = dZ[:, i]
        g["Wo"] += sn.T @ dz
        g["Wco"] += c.T @ dz
        g["bo"] += dz.sum(0)
        dax, ds = gru_back(ds + dz @ p["Wo"].T, gcache, p["Ud"], g["Ud"])
        np.add.at(g["Ey"], Yin[:, i], dax)
        g["bd"] += dax.sum(0)
        g["Wc"] += c.T @ dax
        dc = dz @ p["Wco"].T + dax @ p["Wc"].T
        da = np.einsum("bh,bsh->bs", dc, Hs)  # c = Σ a_j h_j
        dHs += a[:, :, None] * dc[:, None, :]
        de = a * (da - (a * da).sum(1, keepdims=True))  # qua softmax
        g["va"] += np.einsum("bs,bsa->a", de, t)
        dt = de[:, :, None] * p["va"] * (1 - t**2)  # e_j = tanh(keys_j + s W_a) · v_a
        dkeys += dt
        g["Wa"] += s.T @ dt.sum(1)
        ds = ds + dt.sum(1) @ p["Wa"].T  # truy vấn là trạng thái trước bước i
    return dHs, dkeys, ds


def backward(p, cache, dZ):
    """Gradient của mọi tham số khi biết dL/dlogits dZ (B, T, 11) ở mọi bước."""
    X, M, _, Hs, ecache, _ = cache
    g = {k: np.zeros_like(v) for k, v in p.items()}
    dHs, dkeys, dh = backward_decoder(p, g, cache, dZ)
    g["Ua"] += np.einsum("bsh,bsa->ha", Hs, dkeys)
    dHs += dkeys @ p["Ua"].T
    for j in reversed(range(X.shape[1])):  # bộ mã hóa; h_T là trạng thái đầu của bộ giải mã
        dh = dh + dHs[:, j]
        m = M[:, j : j + 1]
        dax, dh_prev = gru_back(m * dh, ecache[j], p["Ue"], g["Ue"])
        np.add.at(g["Ex"], X[:, j], dax)
        g["be"] += dax.sum(0)
        dh = dh_prev + (1 - m) * dh  # phần đệm chỉ chuyển h qua
    return g


def surrogate_grad(P, Y, L, A):
    """dL/dlogits của L = −(1/N) Σ_i A_i log π(y_i): A_i (p_t − onehot(y_t)) / N ở các vị trí được tính."""
    mask = np.arange(Y.shape[1])[None] < L[:, None]
    return A[:, None, None] * (P - np.eye(N_OUT)[Y]) * mask[:, :, None] / len(Y)

Kiểm tra bằng sai phân hữu hạn trên chính mô hình SFT-240: 3 câu hỏi, 3 câu trả lời ngẫu nhiên có độ dài 4, 2, 3 và
các lợi thế $A = (0{,}7;\ -1{,}3;\ 0{,}4)$. Với mỗi ma trận, ta so 3 phần tử có gradient lớn nhất với
$(\mathcal{L}(\theta + \varepsilon) - \mathcal{L}(\theta - \varepsilon)) / 2\varepsilon$.

In [ ]:
rng = np.random.default_rng(0)
Xf, Mf = questions(["150", "2005", "1000001"])
Yf, Lf, Af = rng.integers(0, N_OUT, (3, 4)), np.array([4, 2, 3]), np.array([0.7, -1.3, 0.4])
pf = {k: v.copy() for k, v in MODELS["sft"].items()}


def surrogate(q):
    return -(Af * seq_logp(forward(q, Xf, Mf, Yf)[0], Yf, Lf)[0]).sum() / 3


P_f, cache_f = forward(pf, Xf, Mf, Yf)
g_f = backward(pf, cache_f, surrogate_grad(P_f, Yf, Lf, Af))
worst = 0.0
for k, w in pf.items():
    for flat in np.argsort(-np.abs(g_f[k]).ravel())[:3]:
        idx = np.unravel_index(flat, w.shape)
        old = w[idx]
        w[idx] = old + 1e-5
        up = surrogate(pf)
        w[idx] = old - 1e-5
        down = surrogate(pf)
        w[idx] = old
        num = (up - down) / 2e-5
        worst = max(worst, abs(num - g_f[k][idx]) / max(1e-4, abs(num) + abs(g_f[k][idx])))
print(f"{3 * len(pf)} phần tử, sai số tương đối lớn nhất {sci(worst)}")
assert worst < 1e-6

Gradient viết tay khớp sai phân hữu hạn. Giờ ghép thành một bước học tăng cường, giống `rl.rl` của reel:
1. rút $Q = 16$ số mới (không có nhãn) từ `default_rng(50_000 + seed)`;
2. cho mô hình viết $G = 8$ câu trả lời cho mỗi câu ($T = 1$);
3. chấm bằng bộ kiểm tra, tính $R_i = r_i - \beta \log(\pi_\theta(y_i)/\pi_{\text{ref}}(y_i))$ và lợi thế $A_i$;
4. một bước Adam ($\eta = 10^{-3}$, cắt chuẩn gradient ở 1) trên hàm mất mát thay thế.

Hàm `evaluate` đo trên 600 số kiểm định đúng các cột mà reel lưu mỗi 100 bước: đọc đúng (tham lam), điểm bộ chấm cẩu
thả, $\pi(\text{đúng})$ trung bình, và KL tới $\pi_{\text{ref}}$ (nat). KL của cả câu trả lời được ước lượng bằng
cách lấy một mẫu cho mỗi câu hỏi rồi cộng KL **chính xác** của từng token dọc theo mẫu đó:
$\sum_t \sum_v \pi_\theta(v \mid \dots) \log \frac{\pi_\theta(v \mid \dots)}{\pi_{\text{ref}}(v \mid \dots)}$.

In [ ]:
class Adam:
    """Adam (Bài 07) với cắt chuẩn gradient, như reel."""

    def __init__(self, p):
        self.m = {k: np.zeros_like(v) for k, v in p.items()}
        self.v = {k: np.zeros_like(v) for k, v in p.items()}
        self.t = 0

    def step(self, p, g, lr, clip=1.0):
        self.t += 1
        norm = math.sqrt(sum((x**2).sum() for x in g.values()))
        scale = min(1.0, clip / norm)
        for k in p:
            self.m[k] = 0.9 * self.m[k] + 0.1 * scale * g[k]
            self.v[k] = 0.999 * self.v[k] + 0.001 * (scale * g[k]) ** 2
            m_hat, v_hat = self.m[k] / (1 - 0.9**self.t), self.v[k] / (1 - 0.999**self.t)
            p[k] -= lr * m_hat / (np.sqrt(v_hat) + 1e-8)


def rl_step(p, ref, opt, rng, beta, reward=verify, Q=16, G=8, lr=1e-3):
    """Một bước học tăng cường kiểu GRPO, đơn giản hóa. Trả về phần thưởng (Q, G) của bộ chấm."""
    rows = ["".join(map(str, sample(rng, int(d), 1)[0])) for d in rng.integers(1, 13, Q)]
    Xq, Mq = questions(rows)
    Y = generate(p, Xq, Mq, rng.random((TMAX, Q * G)), rep=G)  # G câu trả lời cho mỗi câu hỏi
    Xg, Mg, rows_g = np.repeat(Xq, G, 0), np.repeat(Mq, G, 0), list(np.repeat(rows, G))
    r, L = reward(rows_g, Y), answer_len(Y)
    P, cache = forward(p, Xg, Mg, Y)
    lp = seq_logp(P, Y, L)[0]
    lq = seq_logp(forward(ref, Xg, Mg, Y)[0], Y, L)[0] if beta else lp  # β = 0: không cần π_ref
    R = (r - beta * (lp - lq)).reshape(Q, G)
    A = R - (R.sum(1, keepdims=True) - R) / (G - 1)  # đường cơ sở: trung bình 7 câu còn lại
    opt.step(p, backward(p, cache, surrogate_grad(P, Y, L, A.ravel())), lr)
    return r.reshape(Q, G)


def kl_sampled(p, ref, X, M, seed):
    """KL(π_θ‖π_ref) của cả câu trả lời (nat), trung bình trên một mẫu cho mỗi câu hỏi."""
    Y = generate(p, X, M, np.random.default_rng(seed).random((TMAX, len(X))))
    Pp, Pq = forward(p, X, M, Y)[0], forward(ref, X, M, Y)[0]
    per_token = (Pp * np.log(Pp / Pq)).sum(-1)
    return float((per_token * (np.arange(TMAX)[None] < answer_len(Y)[:, None])).sum(1).mean())


def evaluate(p, ref, data, seed):
    rows_v, Xv, Mv = data
    out = generate(p, Xv, Mv)
    return {
        "đúng": verify(rows_v, out).mean(),
        "cẩu thả": proxy(rows_v, out).mean(),
        "π(đúng)": np.exp(logprob(p, rows_v, rows_v)).mean(),
        "KL": kl_sampled(p, ref, Xv, Mv, seed),
    }

### 🤔 Dự đoán trước
Ta chạy 100 bước với $\beta = 0{,}1$ từ SFT-240. Mỗi bước, bộ kiểm tra chấm $16 \times 8$ câu trả lời. Có bao nhiêu
phần trăm **nhóm** 8 câu trả lời (cùng một câu hỏi) có cả câu đúng lẫn câu sai, tức là có tín hiệu để học: gần 100%,
khoảng một nửa, hay dưới một phần tư?

In [ ]:
ref = MODELS["sft"]
policy = {k: v.copy() for k, v in ref.items()}
opt, rng = Adam(policy), np.random.default_rng(50_000 + 0)  # seed 0, như reel
mixed = []
for step in range(100):
    rg = rl_step(policy, ref, opt, rng, beta=0.1)
    mixed.append(((rg.max(1) - rg.min(1)) > 0).mean())  # nhóm có cả đúng lẫn sai
before, after = evaluate(ref, ref, VAL, 0), evaluate(policy, ref, VAL, 100)
for key in before:
    f = (lambda v: vn(v, 3)) if key == "KL" else pct
    print(f"{key:>8}: SFT-240 {f(before[key]):>7} → sau 100 bước {f(after[key]):>7}")
print(f"nhóm 8 câu có cả đúng lẫn sai: {pct(np.mean(mixed), 1)} ({sum(round(16 * m) for m in mixed)} trên 1.600 nhóm)")
stored100 = runs["rl"]["b0.1"]["seeds"][0]["curve"][1]  # [bước, đúng, cẩu thả, π(đúng), KL, nhóm lẫn]
# đây là huấn luyện: trên máy khác, sai số làm tròn có thể đổi vài mẫu, nên so với dung sai cỡ vài câu hỏi
assert stored100[0] == 100 and abs(after["đúng"] - stored100[1]) < 5e-3 and abs(after["cẩu thả"] - stored100[2]) < 5e-3
assert abs(after["π(đúng)"] - stored100[3]) < 5e-3 and abs(after["KL"] - stored100[4]) < 0.05, "khác với reel"
assert abs(np.mean(mixed) - stored100[5]) < 5e-3, "khác với reel"

Chỉ khoảng một phần tư số nhóm có cả câu đúng lẫn câu sai. Ở các nhóm còn lại, cả 8 câu đều đúng (câu dễ) hoặc đều sai
(câu quá khó), $r_i$ bằng nhau nên $A_i$ chỉ còn phần chênh của khoản phạt KL: phần thưởng của bộ kiểm tra không dạy gì
ở đó. Chia cho độ lệch chuẩn như GRPO cũng không cứu được, vì $0/0$ được đặt bằng 0. Đây là lý do một số hệ thống học
suy luận bỏ các nhóm toàn đúng hoặc toàn sai khi lập lô và lấy mẫu thêm câu hỏi khác (dynamic sampling của DAPO, Yu
và cộng sự, 2025).

Sau 100 bước (trên tổng 1.000 của reel), tỉ lệ đọc đúng trên tập kiểm định đã tăng và mô hình đã lệch khoảng 0,89 nat
khỏi $\pi_{\text{ref}}$. Các số in ra trùng với hàng “bước 100” mà reel lưu cho đúng lần chạy này (cùng seed, cùng
chuỗi số ngẫu nhiên; trên máy khác có thể lệch một hai câu do làm tròn): bản numpy ở trên chính là thuật toán đã tạo
ra các mô hình của demo.

### Toàn bộ các lần chạy
1.000 bước cho mỗi cấu hình và 3 seed là quá lâu cho notebook. Ô dưới đọc kết quả kiểm tra của mọi lần chạy từ tệp
của demo (1.200 số thử): đọc đúng (tham lam), điểm bộ chấm cẩu thả, đúng khi lấy một mẫu ($\pi(\text{đúng})$ trung bình),
KL tới bản SFT, và tỉ lệ nhóm có cả đúng lẫn sai. Mỗi ô là khoảng thấp nhất–cao nhất của 3 seed.

In [ ]:
ROWS = [  # (nhãn, khóa trong tệp kết quả)
    ("SFT-240", ("sft", "240")),
    ("SFT-960", ("sft", "960")),
    ("SFT-3.840", ("sft", "3840")),
    ("SFT-240 + RL, β = 0", ("rl", "b0.0")),
    ("SFT-240 + RL, β = 0,03", ("rl", "b0.03")),
    ("SFT-240 + RL, β = 0,1", ("rl", "b0.1")),
    ("SFT-240 + RL, β = 0,3", ("rl", "b0.3")),
    ("… chia độ lệch chuẩn, β = 0", ("rl", "norm0.0")),
    ("SFT-960 + RL, β = 0", ("rl", "n960b0.0")),
    ("… bộ chấm cẩu thả, β = 0", ("rl", "proxy0.0")),
    ("… bộ chấm cẩu thả, β = 0,1", ("rl", "proxy0.1")),
]
TABLE = {}
print(f"{'':>28} {'đúng':>15} {'điểm cẩu thả':>15} {'đúng (1 mẫu)':>15} {'KL (nat)':>11} {'nhóm lẫn':>13}")
for label, (kind, key) in ROWS:
    seeds = runs[kind][key]["seeds"]
    t = [s["test"] for s in seeds]
    row = {k: [x[k] for x in t] for k in ("greedy", "greedyProxy", "sample")}
    row["kl"] = [x.get("kl", 0.0) for x in t]
    row["mixed"] = [s.get("mixed", float("nan")) for s in seeds]
    TABLE[key if kind == "rl" else f"sft{key}"] = row
    kl = span(row["kl"], lambda v: vn(v, 2)) if kind == "rl" else "—"
    mx = span(row["mixed"], lambda v: pct(v, 1)) if kind == "rl" else "—"
    print(
        f"{label:>28} {span(row['greedy']):>15} {span(row['greedyProxy']):>15} {span(row['sample']):>15} {kl:>11} {mx:>13}"
    )
below = sum(a < b for a, b in zip(TABLE["proxy0.0"]["greedy"], TABLE["sft240"]["greedy"]))
print(f"học theo bộ chấm cẩu thả (β = 0): {below}/3 seed đọc đúng ít hơn chính bản SFT-240 của seed đó")

Đọc bảng từ trên xuống:
- **Học tăng cường với bộ kiểm tra** đưa SFT-240 từ 53,25–56,00% lên 75,33–78,17% (β = 0), không dùng thêm nhãn nào,
  nhưng lệch 3,57–3,99 nat khỏi bản SFT. Với β = 0,1, mô hình lệch ít hơn hẳn (1,19–1,96 nat) và đúng 67,58–73,25%.
- **Thêm nhãn vẫn tốt hơn ở đây:** SFT trên 960 số đã đạt 85,17–86,50%, hơn mọi lần học tăng cường từ SFT-240. Học tăng
  cường giỏi trau chuốt một mô hình đã khá, không thay được dữ liệu tốt.
- **Chia độ lệch chuẩn** như GRPO cho 68,75–77,67%, so với 75,33–78,17% khi trừ trung bình 7 câu còn lại: hai khoảng
  chồng lên nhau, trong dao động giữa các seed.
- **Bộ chấm cẩu thả** đẩy điểm của chính nó lên 88,00–92,67%, còn tỉ lệ đọc đúng thật chỉ 44,75–59,67%. Mục 10 quay lại
  chuyện này.

Hình dưới vẽ quá trình học trên 600 số kiểm định (đường: trung bình 3 seed, dải: thấp nhất–cao nhất): bên trái là tỉ lệ
đọc đúng, bên phải là KL tới bản SFT. Xanh dương, chấm tròn: β = 0; xanh lá, tam giác, nét đứt: β = 0,1.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.4))
for key, color, marker, ls, label in (("b0.0", BLUE, "o", "-", "β = 0"), ("b0.1", GREEN, "^", "--", "β = 0,1")):
    curves = np.array([s["curve"] for s in runs["rl"][key]["seeds"]], dtype=float)  # (seed, điểm đo, cột)
    steps = curves[0, :, 0]
    for ax, col, scale in ((a1, 1, 100), (a2, 4, 1)):
        v = scale * curves[:, :, col]
        ax.fill_between(steps, v.min(0), v.max(0), color=color, alpha=0.18)
        ax.plot(steps, v.mean(0), marker=marker, ls=ls, color=color, ms=4, label=label)
a1.set(xlabel="bước học tăng cường", ylabel="đọc đúng, tập kiểm định (%)", title="Tỉ lệ đọc đúng tăng…")
a2.set(xlabel="bước học tăng cường", ylabel="KL(π_θ ‖ π_ref), nat", title="…và mô hình rời xa bản SFT")
for ax in (a1, a2):
    ax.grid(alpha=0.3)
    ax.legend(frameon=False)
vn_axes(fig)
plt.show()
for key, label in (("b0.0", "β = 0  "), ("b0.1", "β = 0,1")):
    kls = [c[4] for s in runs["rl"][key]["seeds"] for c in s["curve"][1:]]  # bước 100 … 1.000, 3 seed
    print(f"{label}: KL tới bản SFT ở bước 100–1.000 (3 seed): {vn(min(kls), 2)}–{vn(max(kls), 2)} nat")

Mô hình rời bản SFT ngay trong 100 bước đầu. Với β = 0, không có gì kéo nó về: KL dao động trong khoảng rộng, từ 1,59 tới
5,36 nat. Với β = 0,1, khoản phạt giữ KL ở 0,89–2,61 nat, và tỉ lệ đọc đúng tăng chậm hơn: đó là cái giá của sợi dây.

## 8 · Lời giải đóng của bài toán có phạt KL

Khi tập câu trả lời là hữu hạn, ta giải được bài toán của mục 7 bằng tay. Viết $\pi^*(y) = \pi_{\text{ref}}(y)\,
e^{r(y)/\beta} / Z$ với $Z = \sum_y \pi_{\text{ref}}(y)\, e^{r(y)/\beta}$. Thì

$$J(\pi) = \sum_y \pi(y)\, r(y) - \beta \sum_y \pi(y) \log \frac{\pi(y)}{\pi_{\text{ref}}(y)} = -\beta \sum_y \pi(y)
\log \frac{\pi(y)}{\pi^*(y)} + \beta \log Z = -\beta\, D_{\mathrm{KL}}(\pi \,\|\, \pi^*) + \beta \log Z.$$

KL không âm và bằng 0 chỉ khi $\pi = \pi^*$, nên $\pi^*$ là lời giải tối ưu: lấy $\pi_{\text{ref}}$, nhân mỗi câu với
$e^{r/\beta}$, rồi chuẩn hóa lại. β lớn thì $\pi^*$ gần như giữ nguyên $\pi_{\text{ref}}$; β → 0 thì nó dồn hết xác suất
vào các câu có thưởng cao nhất, nhưng **giữ nguyên tỉ lệ** giữa chúng như trong $\pi_{\text{ref}}$. Không câu nào mà
$\pi_{\text{ref}}$ cho xác suất 0 có thể xuất hiện trong $\pi^*$.

Lấy lại câu “năm nghìn hai trăm”. Tệp `posttrain-cands.json` có xác suất chính xác của mọi câu trả lời hay gặp
(các câu khác nhau trong 512 mẫu của SFT-240) và hai “giỏ” cho phần còn lại: các câu khác mà bộ chấm cẩu thả nhận, và
các câu sai khác. Ô dưới tính $\pi^*$ với vài giá trị β.

In [ ]:
def items(c, model="sft"):
    """Mọi câu trả lời của một câu hỏi ở thẻ 2, rồi hai giỏ: xác suất dưới `model`, phần thưởng r và điểm cẩu thả."""
    labels = [x["y"] for x in c["cands"]] + ["khác, cẩu thả nhận", "khác, sai"]
    pr = [x["p"][model] for x in c["cands"]] + [c["otherProxy"][model], c["other"][model]]
    r = [x["r"] for x in c["cands"]] + [0, 0]
    px = [x["proxy"] for x in c["cands"]] + [1, 0]
    keep = [i for i, v in enumerate(pr) if v > 0]
    return [labels[i] for i in keep], *(np.array([v[i] for i in keep], float) for v in (pr, r, px))


def star(pref, r, beta):
    """π* ∝ π_ref·e^{r/β}, E_π*[r] và KL(π*‖π_ref) (nat)."""
    w = pref * np.exp((r - r.max()) / beta)  # trừ r.max() để β nhỏ không tràn số
    q = w / w.sum()
    return q, q @ r, float((q * np.log(np.where(q > 0, q / pref, 1))).sum())


lab, pref, r5, px5 = items(q5200)
print("π_ref:", " · ".join(f"{a} {pct(v)}" for a, v in zip(lab, pref)))
for beta in (3, 1, 0.3, 0.1, 0.03):
    q, er, kl = star(pref, r5, beta)
    print(f"β = {vn(beta, 2):>4}: π*(5200) = {pct(q[lab.index('5200')]):>7} · KL(π*‖π_ref) = {vn(kl, 3)} nat")
assert abs(pref.sum() - 1) < 1e-6 and abs(pref[lab.index("5200")] - math.exp(lp[1])) < 1e-6

π* đổi phần thưởng lấy độ lệch: muốn đặt 99% xác suất vào câu đúng, nó phải lệch khoảng 1,2 nat khỏi
$\pi_{\text{ref}}$; KL không vượt quá $-\log \pi_{\text{ref}}(5200) = 1{,}23$ nat, mức của việc chỉ còn giữ câu đúng.

Hai kiểm tra trên chính bảng này, ở β = 0,1. Trên tập hữu hạn, mô hình là softmax của các logit $\theta$; gradient
chính xác của $J$ theo $\theta_k$ là $\pi_k\,(R_k - \mathbb{E}_\pi[R])$ với $R = r - \beta \log(\pi/\pi_{\text{ref}})$.
1. Leo dốc theo gradient đó từ $\pi_{\text{ref}}$ hội tụ về đúng $\pi^*$.
2. Ước lượng REINFORCE của mục 7 (nhóm 8 mẫu, đường cơ sở là trung bình 7 mẫu còn lại) không chệch: trung bình của
   20.000 nhóm khớp gradient chính xác, trong vài lần sai số chuẩn.

In [ ]:
beta = 0.1
q_star = star(pref, r5, beta)[0]


def exact_grad(theta):
    pi = softmax(theta)
    R = r5 - beta * np.log(pi / pref)
    return pi * (R - pi @ R), pi, R


theta = np.log(pref).copy()
for _ in range(20_000):
    theta += 5.0 * exact_grad(theta)[0]
print(f"leo dốc từ π_ref: |π − π*| lớn nhất = {sci(np.abs(softmax(theta) - q_star).max())}")
assert np.abs(softmax(theta) - q_star).max() < 1e-4

theta0 = np.log(star(pref, r5, 0.3)[0])  # một điểm giữa đường, nơi khoản phạt KL khác 0
g_exact, pi0, R0 = exact_grad(theta0)
rng = np.random.default_rng(0)
Y = np.minimum((rng.random((20_000, 8, 1)) > np.cumsum(pi0)).sum(2), len(pi0) - 1)  # 20.000 nhóm × 8 mẫu
A = R0[Y] - (R0[Y].sum(1, keepdims=True) - R0[Y]) / 7
g_mc = (A[:, :, None] * (np.eye(len(pi0))[Y] - pi0)).mean(1)  # ∇ log π(y) theo θ = onehot(y) − π
z_score = (g_mc.mean(0) - g_exact) / (g_mc.std(0) / math.sqrt(len(g_mc)))
for a, ge, gm in zip(lab, g_exact, g_mc.mean(0)):
    print(f"  {a:>18}: chính xác {vn(ge, 4):>8} · REINFORCE {vn(gm, 4):>8}")
print(f"{vn(len(g_mc), 0)} nhóm × 8 mẫu; sai khác lớn nhất: {vn(np.abs(z_score).max(), 2)} lần sai số chuẩn")
assert np.abs(z_score).max() < 4

Hai kiểm tra đều qua. Học tăng cường thật làm đúng việc này, chỉ khác là trên hàng tỷ câu trả lời có thể có, với một
mạng nơ-ron dùng chung cho mọi câu hỏi, và với gradient ước lượng từ vài mẫu.

## 9 · DPO: học thẳng từ các cặp tốt–kém

RLHF cổ điển (InstructGPT) học một **mô hình phần thưởng** từ các cặp mà người chọn, rồi chạy học tăng cường. DPO
(Rafailov và cộng sự, 2023) bỏ cả hai bước bằng cách lấy log hai vế của $\pi^*$:

$$r(x, y) = \beta \log \frac{\pi^*(y \mid x)}{\pi_{\text{ref}}(y \mid x)} + \beta \log Z(x).$$

Trong mô hình Bradley–Terry, người chọn $y_w$ hơn $y_l$ với xác suất $\sigma(r(y_w) - r(y_l))$, nên chỉ *hiệu* hai phần
thưởng có mặt và $Z(x)$ triệt tiêu. Thay $\pi^*$ bằng mô hình đang học $\pi_\theta$, ta được một hàm mất mát phân loại
trên các cặp:

$$\mathcal{L}_{\text{DPO}} = -\log \sigma\Big(\beta \log \frac{\pi_\theta(y_w)}{\pi_{\text{ref}}(y_w)} - \beta \log
\frac{\pi_\theta(y_l)}{\pi_{\text{ref}}(y_l)}\Big).$$

Không cần mô hình phần thưởng riêng, không cần lấy mẫu trong lúc học, nhưng vẫn cần $\pi_{\text{ref}}$:
$\beta \log(\pi_\theta/\pi_{\text{ref}})$ chính là **phần thưởng ngầm** mà DPO học.

Ta thử trên bảng của “năm nghìn hai trăm”, như demo: rút các cặp $y_1, y_2 \sim \pi_{\text{ref}}$ (rút lại $y_2$ đến
khi khác $y_1$), rồi cho “người chọn” giả chọn $y_1$ với xác suất $\sigma(r(y_1) - r(y_2))$. Bộ số ngẫu nhiên là
mulberry32 của demo (giống Bài 16), để các cặp trùng với các cặp trong trình duyệt.

In [ ]:
def mulberry32(seed, n):
    """n số đều trong [0, 1) từ mulberry32(seed), giống hệt hàm JavaScript của demo."""
    a = ((seed + np.arange(1, n + 1, dtype=np.uint64) * 0x6D2B79F5) & 0xFFFFFFFF).astype(np.uint32)
    t = (a ^ (a >> 15)) * (a | 1)
    t = (t + (t ^ (t >> 7)) * (t | 61)) ^ t
    return (t ^ (t >> 14)).astype(np.float64) / 2**32


def pairs(pref, r, n, seed):
    """n cặp ưu tiên, như demo: W[w, l] là số lần y_w thắng y_l."""
    size = 4 * n
    while True:
        u = mulberry32(seed, size)
        pick = np.minimum(np.searchsorted(np.cumsum(pref), u, side="right"), len(pref) - 1)  # rút theo π_ref
        W, k = np.zeros((len(pref), len(pref))), 0
        try:
            for _ in range(n):
                i1, i2, k, tries = pick[k], pick[k + 1], k + 2, 1
                while i2 == i1:  # rút lại y₂ (sau 1.000 lần thì lấy mục kế tiếp)
                    if tries >= 1000:
                        i2 = (i1 + 1) % len(pref)
                        break
                    i2, k, tries = pick[k], k + 1, tries + 1
                win = u[k] < sig(r[i1] - r[i2])  # Bradley–Terry
                W[(i1, i2) if win else (i2, i1)] += 1
                k += 1
            return W
        except IndexError:  # dãy số chưa đủ dài: lấy dãy dài gấp đôi (cùng phần đầu)
            size *= 2


def dpo_fit(W, pref, beta, ref=True, steps=3000, lr=0.5):
    """DPO trên bảng: π_θ = softmax(θ), bắt đầu từ π_ref. Viết theo v = β(θ − log π_ref) thì lề của một cặp là
    v_w − v_l (có π_ref); không có π_ref, lề là β log π_θ(y_w)/π_θ(y_l). Hạ gradient toàn lô trên −log σ(lề)."""
    off = np.zeros(len(pref)) if ref else beta * np.log(pref)
    v = np.zeros(len(pref))
    for _ in range(steps):
        margin = (v + off)[:, None] - (v + off)[None, :]
        d = W * (1 - sig(margin))  # −∂L/∂lề, cho mọi cặp (w, l)
        v -= lr * beta**2 * (d.sum(0) - d.sum(1)) / W.sum()
    return v


def implicit(v, pref, beta):
    """Chính sách đã khớp π ∝ π_ref·e^{v/β} và phần thưởng ngầm β log(π/π_ref)."""
    pi = pref * np.exp((v - v.max()) / beta)
    pi /= pi.sum()
    return beta * np.log(pi / pref), pi


W = pairs(pref, r5, 3000, 0)
print(f"3.000 cặp, seed 0: số lần thắng (hàng thắng cột), các mục {lab[:4]}:")
print(W[:4, :4].astype(int))
i_w, i_l = lab.index("5200"), lab.index("520")
print(
    f"“5200” thắng “520” {vn(W[i_w, i_l], 0)} lần, thua {vn(W[i_l, i_w], 0)} lần: "
    f"{pct(W[i_w, i_l] / (W[i_w, i_l] + W[i_l, i_w]))}; σ(1) = {pct(sig(1.0))}"
)

Câu đúng “5200” thắng “520” 2.166 lần và thua 765 lần, gần đúng tỉ lệ $\sigma(1)$ của Bradley–Terry với hiệu phần
thưởng 1. Ô dưới khớp DPO ($\beta = 0{,}1$) với 30, 300 và 3.000 cặp, 3 seed mỗi cỡ, có và không có
$\pi_{\text{ref}}$ trong lề. Rồi nó so phần thưởng ngầm với phần thưởng thật $r$, sau khi trừ mỗi bên trung bình (theo
$\pi_{\text{ref}}$) của nó trên các câu có mặt trong ít nhất một cặp, vì phần thưởng chỉ xác định sai khác một hằng số.
Cột “sai lệch” là trung bình có trọng số $\pi_{\text{ref}}$ của $|$phần thưởng ngầm − $r|$, trên thang mà câu đúng
và câu sai cách nhau 1, như demo.

In [ ]:
def dpo_report(pref, r, n, seed, beta=0.1, ref=True):
    """Khớp DPO trên n cặp; trả về sai lệch trung bình của phần thưởng ngầm, E_π[r] và KL(π‖π_ref)."""
    W = pairs(pref, r, n, seed)
    imp, pi = implicit(dpo_fit(W, pref, beta, ref), pref, beta)
    seen = (W.sum(0) + W.sum(1)) > 0  # các câu có mặt trong ít nhất một cặp
    w = pref[seen].sum()
    err = np.abs((imp - (pref * imp)[seen].sum() / w) - (r - (pref * r)[seen].sum() / w))
    return (pref * err)[seen].sum() / w, pi @ r, float((pi * np.log(pi / pref)).sum())


dpo = {(n, w): [dpo_report(pref, r5, n, s, ref=w) for s in range(3)] for n in (30, 300, 3000) for w in (1, 0)}
print(f"π* (β = 0,1): đúng {pct(star(pref, r5, 0.1)[1])}, KL {vn(star(pref, r5, 0.1)[2], 3)} nat")
for (n, with_ref), res in dpo.items():
    err, er, kl = zip(*res)
    print(
        f"{vn(n, 0):>5} cặp, {'có π_ref  ' if with_ref else 'không π_ref'}: sai lệch {span(err, lambda v: vn(v, 3)):>13}"
        f" · chính sách đúng {span(er):>15} · KL {span(kl, lambda v: vn(v, 3))} nat"
    )
assert max(e for e, _, _ in dpo[3000, 1]) < 0.08 and min(e for _, e, _ in dpo[3000, 1]) > 0.99

Với 3.000 cặp, DPO có $\pi_{\text{ref}}$ tìm lại gần đúng $\pi^*$: chính sách đúng trên 99%, KL gần 1,2 nat như $\pi^*$,
và phần thưởng ngầm khớp $r$ với sai lệch nhỏ ở cả 3 seed. Câu này dễ cho DPO: gần như mọi cặp là “520” đấu “5200”,
nên 30 cặp đã đủ để chọn đúng; chỉ có sai lệch là dao động nhiều hơn giữa các seed, vì các câu hiếm không xuất hiện
trong cặp nào và DPO không biết gì về chúng.

Bỏ $\pi_{\text{ref}}$ khỏi lề (lề thành $\beta \log \pi_\theta(y_w)/\pi_\theta(y_l)$) vẫn cho chính sách tốt trên bài
nhỏ này, nhưng ý nghĩa đã khác: “phần thưởng ngầm” không còn là thay đổi so với mô hình gốc, và không còn gì giữ
mô hình gần $\pi_{\text{ref}}$. Trên cả 12 câu hỏi của thẻ 2 (3 seed mỗi câu), tệp kết quả của demo cho thấy số cặp
quan trọng thế nào khi có nhiều câu trả lời cạnh tranh:

In [ ]:
for d in cands["dpo"]:
    print(
        f"{vn(d['n'], 0):>5} cặp, {'có π_ref  ' if d['ref'] else 'không π_ref'}: sai lệch "
        f"{vn(d['mean'][0], 3)}–{vn(d['mean'][1], 3)} · đúng {pct(d['er'][0])}–{pct(d['er'][1])}"
    )

Với 30 cặp, có câu hỏi mà chính sách DPO chỉ đúng 26,62%; với 3.000 cặp, mọi câu đều trên 99%. DPO cần đủ cặp phủ
những câu trả lời quan trọng, vì nó chỉ học được từ những gì có trong dữ liệu ưu tiên.

## 10 · Goodhart: khi thước đo thành mục tiêu

“Khi một thước đo trở thành mục tiêu, nó không còn là thước đo tốt” (định luật Goodhart). Gao, Schulman & Hilton (2022)
đo điều này với mô hình phần thưởng thật: tối ưu theo một mô hình phần thưởng thay thế, điểm “vàng” (do một mô hình
lớn hơn chấm) tăng rồi giảm khi KL tới mô hình ban đầu tăng. Bài toán đọc số cho ta xem cùng hiện tượng một cách chính
xác, vì ta có cả thước đo thật (`verify`) lẫn thước đo hỏng (`proxy`).

Bắt đầu với lời giải đóng. Nếu phần thưởng là `proxy`, $\pi^*$ dồn xác suất vào **mọi** câu mà bộ chấm cẩu thả nhận,
và giữ nguyên tỉ lệ giữa chúng như $\pi_{\text{ref}}$. Với “năm nghìn hai trăm”, $\pi_{\text{ref}}$ ưa “520” gấp đôi
“5200” hơn gấp đôi, nên tối ưu bộ chấm cẩu thả không bao giờ làm câu đúng thắng:

In [ ]:
for beta in (1, 0.3, 0.1, 0.03):
    q_r, q_px = star(pref, r5, beta)[0], star(pref, px5, beta)[0]
    print(
        f"β = {vn(beta, 2):>4}: tối ưu theo bộ kiểm tra → đúng {pct(q_r @ r5):>7}"
        f" · tối ưu theo bộ cẩu thả → điểm cẩu thả {pct(q_px @ px5):>7}, đúng thật {pct(q_px @ r5):>7}"
    )
ceiling_5200 = pref[lab.index("5200")] / (pref * px5).sum()  # phần của câu đúng trong các câu bộ cẩu thả nhận
print(f"β → 0, bộ cẩu thả: đúng thật tiến tới π_ref(5200) / π_ref(cẩu thả nhận) = {pct(ceiling_5200)}")

Trên cả 1.200 câu thử cũng vậy: tối ưu hoàn hảo theo bộ cẩu thả (β → 0) cho tỉ lệ đúng bằng trung bình của
$\pi_{\text{ref}}(\text{đúng}) / \pi_{\text{ref}}(\text{cẩu thả nhận})$. Ô dưới tính nó từ $\pi_{\text{ref}}(\text{đúng})$
vừa tính ở mục 6 và khối lượng mà bộ cẩu thả nhận dưới $\pi_{\text{ref}}$ (tệp kết quả lưu sẵn; reel tính nó chính xác
bằng cách duyệt cây mọi câu trả lời mà bộ cẩu thả nhận).

In [ ]:
pm_ref = np.array(runs["ref"]["pm"])  # π_ref(bộ cẩu thả nhận câu trả lời), mỗi câu thử
# câu đúng luôn nằm trong các câu bộ cẩu thả nhận; phép duyệt cây bỏ các nhánh có xác suất dưới 10⁻¹⁰
assert (pm_ref >= pc["sft"] * (1 - 1e-4) - 1e-10).all()
share = np.where(pm_ref > 0, pc["sft"] / np.maximum(pm_ref, 1e-300), 0.0)  # phần của câu đúng trong khối đó
print(f"SFT-240: π(đúng) {pct(pc['sft'].mean())} · π(cẩu thả nhận) {pct(pm_ref.mean())}")
print(f"tối ưu hoàn hảo theo bộ cẩu thả (β → 0): điểm cẩu thả 100%, đúng thật {pct(share.mean())}")
print(
    f"tối ưu hoàn hảo theo bộ kiểm tra (β → 0): đúng {pct(np.mean(pc['sft'] > 0))}, với KL trung bình"
    f" −log π_ref(đúng) = {vn(-np.log(pc['sft']).mean(), 2)} nat"
)

Ngay cả tối ưu hoàn hảo theo bộ cẩu thả cũng chỉ đúng 61,85%, còn mô hình học tăng cường thật thì kém hơn thế. (Tối
ưu hoàn hảo theo bộ kiểm tra thì đúng 100,00%, với cái giá là lệch trung bình 5,44 nat.) Hình
dưới vẽ ba lần chạy học theo bộ cẩu thả với β = 0, trên 600 số kiểm định (đường: trung bình 3 seed, dải: thấp nhất–cao
nhất). Cam, ô vuông, nét đứt: điểm của bộ cẩu thả; cam, ô vuông, nét liền: tỉ lệ đọc đúng thật của cùng mô hình;
xanh dương, chấm tròn: tỉ lệ đọc đúng của mô hình học theo bộ kiểm tra chính xác. Đường xám là SFT-240 (trung bình 3
seed).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.6))
for key, col, color, marker, ls, label in (
    ("proxy0.0", 2, ORANGE, "s", "--", "bộ cẩu thả: điểm của chính nó"),
    ("proxy0.0", 1, ORANGE, "s", "-", "bộ cẩu thả: đọc đúng thật"),
    ("b0.0", 1, BLUE, "o", "-", "bộ kiểm tra: đọc đúng thật"),
):
    curves = 100 * np.array([[c[col] for c in s["curve"]] for s in runs["rl"][key]["seeds"]])
    steps = [c[0] for c in runs["rl"][key]["seeds"][0]["curve"]]
    ax.fill_between(steps, curves.min(0), curves.max(0), color=color, alpha=0.15)
    ax.plot(steps, curves.mean(0), marker=marker, ls=ls, color=color, ms=4, label=label)
ax.axhline(100 * np.mean([s["val"] for s in runs["sft"]["240"]["seeds"]]), color=GREY, lw=1, label="SFT-240")
ax.set(xlabel="bước học tăng cường (β = 0)", ylabel="tập kiểm định (%)", ylim=(30, 100))
ax.set_title("Điểm của bộ chấm cẩu thả tăng, số đọc đúng thật thì không")
ax.legend(frameon=False, fontsize=8, loc="center left", bbox_to_anchor=(1.01, 0.5))
ax.grid(axis="y", alpha=0.3)
vn_axes(fig)
plt.show()
p0 = runs["rl"]["proxy0.0"]["seeds"]
print("KL tới bản SFT ở bước 1.000 (3 seed):", " · ".join(vn(s["curve"][-1][4], 2) for s in p0), "nat")

Điểm của bộ cẩu thả (nét đứt) tăng gần như suốt quá trình, còn tỉ lệ đọc đúng thật (nét liền cam) dao động quanh mức
của SFT rồi tụt xuống ở hai seed, trong khi KL lên tới hơn 10 nat ở một seed. Ba điều đáng nhớ:
- **Phần thưởng hỏng không báo lỗi.** Nếu chỉ nhìn điểm của bộ chấm, lần chạy này trông như thành công.
- **Phạt KL giảm thiệt hại nhưng không sửa được thước đo sai.** Với β = 0,1, tỉ lệ đúng thật giữ ở 62,50–65,58%,
  vẫn kém học theo bộ kiểm tra chính xác cùng β (67,58–73,25%).
- **Phần thưởng kiểm chứng được** (một chương trình chấm đúng–sai, như cho toán và lập trình) khó lách hơn nhiều.
  Đó là lý do DeepSeek-R1-Zero học chỉ với thưởng đúng–sai và thưởng định dạng, không dùng mô hình phần thưởng học
  từ người. Nhưng “kiểm chứng được” chỉ an toàn khi bộ kiểm tra thật sự đúng: `proxy` cũng là một chương trình.

## 11 · Tính toán lúc trả lời: nhiều mẫu, bỏ phiếu, best-of-N

Thay vì học thêm, ta có thể cho mô hình **nghĩ nhiều hơn lúc trả lời**: lấy $k$ mẫu cho mỗi câu hỏi rồi
- **bỏ phiếu đa số** (self-consistency, Wang và cộng sự, 2022): trả lời câu xuất hiện nhiều nhất, hòa thì lấy câu
  xuất hiện trước; không cần bộ kiểm tra;
- **best-of-N với bộ kiểm tra:** trả lời một mẫu mà bộ kiểm tra nhận, nếu có. Với bộ kiểm tra không sai, tỉ lệ đúng
  là “có ít nhất một mẫu đúng trong $k$”, tức pass@k (Chen và cộng sự, 2021, ước lượng không chệch
  $1 - \binom{n-c}{k} / \binom{n}{k}$ từ $n$ mẫu có $c$ mẫu đúng).

Ô dưới lấy 64 mẫu ($T = 1$) cho mỗi câu trong 1.200 câu thử, với đúng dãy số ngẫu nhiên mà reel dùng
(`default_rng(9_000)`, rút theo khối 8 mẫu × 4 lô × 13 bước × 2.400 hàng), cho SFT-240 và mô hình học theo bộ kiểm
tra.
Nhiều mẫu có chung phần đầu (“52…”), nên hàm dưới chỉ tính mỗi trạng thái (câu hỏi, phần đầu) một lần.

In [ ]:
def generate_many(p, X, M, U, rep):
    """Như generate(…, U, rep), nhưng các hàng có cùng câu hỏi và cùng phần đầu dùng chung một lần tính."""
    Hs, h_T, _ = encode(p, X, M)
    keys = Hs @ p["Ua"]
    n = len(X) * rep
    out, act = np.full((n, TMAX), EOS), np.arange(n)
    node = np.repeat(np.arange(len(X)), rep)  # trạng thái hiện tại của mỗi hàng
    s, y, q = h_T, np.full(len(X), BOS), np.arange(len(X))  # mỗi trạng thái: GRU, token vừa viết, câu hỏi
    for i in range(TMAX):
        s, z, _ = logits(p, s, y, Hs[q], keys[q], M[q])
        cum = softmax(z).cumsum(1)[node[act]]
        tok = np.minimum((cum < U[i, act][:, None]).sum(1), N_OUT - 1)
        out[act, i] = tok
        keep = tok != EOS
        act, tok, parent = act[keep], tok[keep], node[act][keep]
        if not len(act):
            break
        uniq, node[act] = np.unique(parent * N_OUT + tok, return_inverse=True)  # các phần đầu mới, khác nhau
        s, y, q = s[uniq // N_OUT], uniq % N_OUT, q[uniq // N_OUT]
    return out


K, n_q = 64, len(rows)
u = np.random.default_rng(9_000).random((K // 8, 4, TMAX, 2400))  # khối 8 mẫu, lô 2.400 hàng, bước, hàng
qi, j = np.repeat(np.arange(n_q), K), np.tile(np.arange(K), n_q)
U64 = u[j // 8, (qi * 8 + j % 8) // 2400, :, (qi * 8 + j % 8) % 2400].T  # (13, 1.200 · 64)
print("dãy số ngẫu nhiên của reel:", " × ".join(vn(v, 0) for v in u.shape))
TTC = {}
for name in ("sft", "rl"):
    out = generate_many(MODELS[name], X, M, U64, K)
    TTC[name] = {
        "ok": verify(list(np.repeat(rows, K)), out).reshape(n_q, K).astype(bool),
        "px": proxy(list(np.repeat(rows, K)), out).reshape(n_q, K).astype(bool),
        "ans": np.array(text(out)).reshape(n_q, K),
    }
    print(f"{NAME[name]:>16}: {vn(n_q * K, 0)} mẫu, {pct(TTC[name]['ok'].mean())} đúng")

Từ các mẫu đó ta tính hai đường theo $k$. Bỏ phiếu được đo như reel: chia 64 mẫu của mỗi câu thành $64/k$ khối rời
nhau gồm $k$ mẫu, rồi lấy trung bình trên các khối.

### 🤔 Dự đoán trước
Với SFT-240 (một mẫu đúng 51,40%), bỏ phiếu đa số trên 64 mẫu lên tới đâu: chỉ nhích vài điểm, khoảng 70%, hay gần
90%? Còn “có ít
nhất một mẫu đúng trong 64”?

In [ ]:
def pass_at_k(ok, k):
    """Ước lượng không chệch của Chen và cộng sự (2021): 1 − C(n − c, k) / C(n, k), trung bình trên các câu hỏi."""
    n = ok.shape[1]
    return float(np.mean([1 - math.comb(n - c, k) / math.comb(n, k) for c in ok.sum(1)]))


def majority_at_k(ans, ok, k):
    """Bỏ phiếu đa số trên các khối k mẫu rời nhau; hòa thì lấy câu xuất hiện trước trong khối."""
    ids = np.array([np.unique(a, return_inverse=True)[1] for a in ans]).reshape(len(ans), -1, k)
    votes = (ids[:, :, :, None] == ids[:, :, None, :]).sum(-1)  # số phiếu của câu ở mỗi vị trí
    pick = votes.argmax(-1)  # vị trí đầu tiên có nhiều phiếu nhất
    return float(np.take_along_axis(ok.reshape(len(ok), -1, k), pick[..., None], -1).mean())


KS = [1, 2, 4, 8, 16, 32, 64]
curves = {
    name: {"pass": [pass_at_k(t["ok"], k) for k in KS], "maj": [majority_at_k(t["ans"], t["ok"], k) for k in KS]}
    for name, t in TTC.items()
}
for name, key in (("sft", "sft240"), ("rl", "rl0")):
    c = curves[name]
    print(
        f"{NAME[name]:>16}: bỏ phiếu {pct(c['maj'][0])} → {pct(c['maj'][-1])}"
        f" · có bộ kiểm tra (pass@k) {pct(c['pass'][0])} → {pct(c['pass'][-1])}  (k = 1 → 64)"
    )
    assert np.allclose(c["pass"], runs["ttc"][key]["passk"], atol=2e-3), "khác với demo"
    assert np.allclose(c["maj"], runs["ttc"][key]["maj"], atol=2e-3), "khác với demo"

Bỏ phiếu đưa SFT-240 từ 51,40% lên 56,00% rồi đứng yên từ khoảng $k = 16$: khi câu sai hay gặp nhất đông hơn câu đúng
(như “520” so với “5200”), thêm phiếu chỉ làm câu sai thắng chắc hơn. Có bộ kiểm tra thì “có ít nhất một mẫu đúng” lên
70,08%: phần chênh 14 điểm chỉ dùng được khi có cách *chọn* ra mẫu đúng.

Thêm mẫu chỉ giúp ở câu mà mô hình **đôi khi** đã làm đúng. Ô dưới đếm, theo số chữ số, số câu (trên 100) không có
mẫu đúng nào trong 64, và đổi chiều: bao nhiêu câu SFT-240 không bao giờ làm đúng mà mô hình học tăng cường làm đúng
được, và ngược lại.

In [ ]:
c_sft, c_rl = TTC["sft"]["ok"].sum(1), TTC["rl"]["ok"].sum(1)
for name, c in (("sft", c_sft), ("rl", c_rl)):
    print(
        f"{NAME[name]:>16}: không mẫu đúng nào trong 64, d = 1…12:",
        [int((c[digits == d] == 0).sum()) for d in range(1, 13)],
    )
agg = {
    "unsolvedBySft": int((c_sft == 0).sum()),
    "solvedByRl": int(((c_sft == 0) & (c_rl > 0)).sum()),
    "lostByRl": int(((c_sft > 0) & (c_rl == 0)).sum()),
}
print(
    f"SFT-240 không làm đúng lần nào: {agg['unsolvedBySft']} câu; học tăng cường làm đúng được {agg['solvedByRl']} câu"
    f" trong số đó, và mất {agg['lostByRl']} câu mà SFT-240 đôi khi làm đúng"
)
assert agg == {k: runs["agg"][k] for k in agg}, "khác với demo"
assert [int((c_sft[digits == d] == 0).sum()) for d in range(1, 13)] == runs["ttc"]["sft240"]["byDigit"]["unsolved"]

Với số 12 chữ số, 88 trong 100 câu không có mẫu đúng nào trong 64 mẫu của SFT-240. Học tăng cường làm được một phần
trong số đó, nhưng cũng làm mất 40 câu mà SFT-240 đôi khi làm đúng.

Mỗi mẫu được chấm, nên ở $k$ rất lớn ta không cần lấy mẫu: với $\pi(\text{đúng})$ chính xác của mục 6, xác suất có
ít nhất một mẫu đúng trong $k$ mẫu là $1 - (1 - \pi(\text{đúng}))^k$. Hình dưới vẽ cả hai cách đo cho SFT-240 (cam, ô
vuông) và mô hình học tăng cường (xanh, chấm tròn). Trái: $k = 1 \dots 64$ từ các mẫu thật, nét liền là có bộ kiểm
tra, nét đứt là bỏ phiếu. Phải: $1 - (1 - \pi)^k$ tính chính xác tới $k = 2^{20}$ (trục log).

In [ ]:
k_big = np.logspace(0, 20 * math.log10(2), 61)
exact = {name: [float(np.mean(-np.expm1(k * np.log1p(-pc[name])))) for k in k_big] for name in ("sft", "rl")}
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
for name, color, marker in (("sft", ORANGE, "s"), ("rl", BLUE, "o")):
    x = np.log2(KS)
    a1.plot(x, 100 * np.array(curves[name]["pass"]), marker=marker, color=color, label=f"{NAME[name]}: có bộ kiểm tra")
    a1.plot(
        x,
        100 * np.array(curves[name]["maj"]),
        marker=marker,
        ls="--",
        color=color,
        mfc="white",
        label=f"{NAME[name]}: bỏ phiếu",
    )
    a2.plot(k_big, 100 * np.array(exact[name]), color=color, marker=marker, markevery=6, label=NAME[name])
a1.set_xticks(np.log2(KS), labels=[str(k) for k in KS])
a1.set(xlabel="số mẫu k", ylabel="đúng (%)", ylim=(45, 100), title="64 mẫu thật")
a2.set(
    xscale="log",
    xlabel="số mẫu k (thang log)",
    ylabel="có ít nhất một mẫu đúng (%)",
    ylim=(45, 90),
    title="Tính chính xác từ π(đúng)",
)
a1.legend(frameon=False, fontsize=7.5, loc="upper left", ncol=2)
a2.legend(frameon=False, fontsize=8, loc="lower right")
for ax in (a1, a2):
    ax.grid(alpha=0.3)
vn_axes(fig)
plt.show()
big = {name: float(np.mean(-np.expm1(2**20 * np.log1p(-pc[name])))) for name in ("sft", "rl")}
cross = next(k for k, s, r in zip(k_big, exact["sft"], exact["rl"]) if s > r)
print(f"k = 2²⁰: SFT-240 {pct(big['sft'], 1)} · RL {pct(big['rl'], 1)}; SFT-240 vượt lên từ khoảng k = {vn(cross, 0)}")
for name, key in (("sft", "sft240"), ("rl", "rl0")):
    assert abs(big[name] - runs["ttc"][key]["passExact"][-1]["acc"]) < 1e-3, "khác với demo"

Hai đường cắt nhau. Học tăng cường làm một mẫu đúng hơn hẳn (75,79% so với 51,47%), hơn cả 64 mẫu có bộ kiểm tra của
SFT-240. Nhưng ở $k$ khổng lồ, SFT-240 lại nhỉnh hơn: học tăng cường dồn xác suất vào những câu nó đã hay làm đúng, và
đẩy xác suất của đáp án ở một số câu khác xuống gần 0. Yue và cộng sự (2025) thấy đúng điều này ở các mô hình suy luận
thật: pass@1 tăng, pass@k ở $k$ lớn thì mô hình nền thắng. Học tăng cường ở đây chủ yếu **mài sắc** những gì mô hình
đã biết, chứ không dạy nó điều mới.

## 12 · Best-of-N có độ lệch KL bao nhiêu?

Best-of-N cũng là một chính sách mới: lấy $N$ mẫu từ $\pi_{\text{ref}}$, trả lời mẫu mà bộ kiểm tra nhận. Nó lệch khỏi
$\pi_{\text{ref}}$ bao nhiêu? Nhiều bài báo dùng công thức $\log N - (N-1)/N$. Beirami và cộng sự (2024) chứng minh đó
chỉ là **cận trên**: nó đúng khi $N$ mẫu gần như luôn khác nhau, và có thể rất lỏng khi các câu trả lời hay trùng nhau.

Với bộ kiểm tra đúng–sai và một đáp án duy nhất, ta tính được KL chính xác. Gọi $p = \pi_{\text{ref}}(\text{đúng})$:
best-of-N trả lời đúng với xác suất $a = 1 - (1-p)^N$; khi không mẫu nào đúng, nó trả một câu sai theo đúng tỉ lệ của
$\pi_{\text{ref}}$. Nên

$$D_{\mathrm{KL}}\big(\pi_{\text{BoN}} \,\|\, \pi_{\text{ref}}\big) = a \log \frac{a}{p} + (1 - a) \log \frac{1 - a}{1 - p}
\;\le\; -\log p.$$

In [ ]:
def bon_kl(p, N):
    """Best-of-N với bộ kiểm tra đúng–sai: xác suất đúng a và KL chính xác tới π_ref (nat), cho mỗi câu hỏi."""
    a = -np.expm1(N * np.log1p(-p))  # 1 − (1 − p)^N, chính xác cả khi p rất nhỏ
    with np.errstate(divide="ignore", invalid="ignore"):
        kl = np.where(a > 0, a * np.log(a / p), 0) + np.where(a < 1, (1 - a) * np.log((1 - a) / (1 - p)), 0)
    return a, kl


def star_binary(p, beta):
    """π*_β với phần thưởng đúng–sai: xác suất đúng q = p / (p + (1 − p)e^{−1/β}) và KL(π*‖π_ref), cho mỗi câu."""
    den = p + (1 - p) * math.exp(-1 / beta)
    q = p / den
    return q, np.maximum(-(1 - q) / beta - np.log(den), 0)


def star_at_kl(p, kl):
    """Độ chính xác của π*_β tại cùng KL trung bình: chia đôi theo log β (KL giảm khi β tăng)."""
    lo, hi = -3.0, 3.0
    for _ in range(60):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if star_binary(p, 10**mid)[1].mean() > kl else (lo, mid)
    return float(star_binary(p, 10**hi)[0].mean())


p_ref = np.clip(pc["sft"], 1e-300, 1 - 1e-16)
for N in KS:
    a, kl = bon_kl(p_ref, N)
    bound = math.log(N) - (N - 1) / N
    stored = next(b for b in runs["ttc"]["sft240"]["bon"] if b["N"] == N)
    print(
        f"N = {N:>2}: đúng {pct(a.mean())} · KL chính xác {vn(kl.mean(), 3)} nat · công thức {vn(bound, 3)} nat"
        f" · π* cùng KL: đúng {pct(star_at_kl(p_ref, kl.mean()))}"
    )
    assert (kl <= bound + 1e-12).all() and (kl <= -np.log(p_ref) + 1e-9).all()  # cận trên, cho từng câu hỏi
    assert abs(a.mean() - stored["acc"]) < 2e-4 and abs(kl.mean() - stored["kl"]) < 2e-4, "khác với demo"
a64, kl64 = bon_kl(p_ref, 64)
print(f"N = 64: công thức lớn gấp {vn((math.log(64) - 63 / 64) / kl64.mean(), 1)} lần KL thật")

Ở $N = 64$, best-of-N đúng 70,26% mà chỉ lệch 0,412 nat khỏi $\pi_{\text{ref}}$, còn công thức cho 3,175 nat. Công
thức giả định $N$ mẫu gần như luôn khác nhau. Ở đây chúng trùng nhau rất nhiều: với câu dễ ($p$ gần 1), best-of-N gần
như không đổi gì, và với câu mà SFT-240 không bao giờ đúng ($p \approx 0$) thì cũng vậy. KL của nó không vượt quá
$-\log p$ ở từng câu, dù $N$ lớn đến đâu.

Cột cuối so với $\pi^*_\beta$ ở **cùng** KL trung bình (chọn β bằng chia đôi): best-of-N gần như tối ưu, chỉ kém
$\pi^*$ vài phần mười điểm. Hình dưới đặt mọi chính sách lên cùng một mặt phẳng: tỉ lệ đúng của một mẫu theo KL tới
SFT-240 (seed 0). Đường xanh lá là $\pi^*_\beta$ khi β chạy từ 10 xuống 0,02, giới hạn trên của mọi chính sách ở cùng
KL. Cam, ô vuông: best-of-N, $N = 1 \dots 64$, đặt ở KL chính xác (ô đặc) và ở KL theo công thức (ô rỗng, nét đứt).
Xanh dương, chấm tròn: bốn lần học tăng cường thật với bộ kiểm tra (seed 0, β = 0; 0,03; 0,1; 0,3; đọc từ tệp kết quả).

In [ ]:
betas = np.logspace(1, math.log10(0.02), 40)
frontier = np.array([[v.mean() for v in star_binary(p_ref, b)[::-1]] for b in betas])  # (KL, đúng)
bon = np.array([[bon_kl(p_ref, N)[1].mean(), bon_kl(p_ref, N)[0].mean()] for N in KS])
rl_pts = [(b, runs["rl"][f"b{b}"]["seeds"][0]["test"]) for b in (0.0, 0.03, 0.1, 0.3)]
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(frontier[:, 0], 100 * frontier[:, 1], color=GREEN, lw=2, label="π*_β (giới hạn trên)")
ax.plot(bon[:, 0], 100 * bon[:, 1], "s-", color=ORANGE, label="best-of-N, KL chính xác")
ax.plot(
    [math.log(N) - (N - 1) / N for N in KS],
    100 * bon[:, 1],
    "s--",
    color=ORANGE,
    mfc="white",
    label="best-of-N, KL theo công thức",
)
ax.plot(
    [t["kl"] for _, t in rl_pts],
    [100 * t["sample"] for _, t in rl_pts],
    "o",
    color=BLUE,
    ms=7,
    label="học tăng cường thật",
)
for b, t in rl_pts:
    ax.annotate(
        f"β = {vn_tick(b)}",
        (t["kl"], 100 * t["sample"]),
        xytext=(6, -12),
        textcoords="offset points",
        fontsize=8,
        color=BLUE,
    )
ax.set(xlabel="KL tới SFT-240 (nat)", ylabel="đúng khi lấy một mẫu (%)", xlim=(0, 4.5), ylim=(50, 100))
ax.set_title("Đổi độ lệch lấy độ chính xác: π*, best-of-N và học tăng cường")
ax.legend(frameon=False, fontsize=8, loc="lower right")
ax.grid(alpha=0.3)
vn_axes(fig)
plt.show()
for b, t in rl_pts:
    print(
        f"RL β = {vn_tick(b)}: đúng {pct(t['sample'])} ở KL {vn(t['kl'], 2)} nat · π* cùng KL: "
        f"{pct(star_at_kl(p_ref, t['kl']))}"
    )
assert abs(star_at_kl(p_ref, rl_pts[2][1]["kl"]) - runs["rl"]["b0.1"]["seeds"][0]["starAtKl"]) < 1e-3, "khác với demo"

Ba điều hình này nói:
- **Mọi chính sách đều đổi KL lấy độ chính xác**, và $\pi^*$ là giới hạn trên: không chính sách nào đúng nhiều hơn ở
  cùng KL. Best-of-N bám sát giới hạn đó (với một bộ kiểm tra không sai) mà không cần học gì, nhưng tốn $N$ lần tính
  toán cho mỗi câu trả lời.
- **Học tăng cường thật nằm dưới đường $\pi^*$ khá xa.** $\pi^*$ chọn riêng cho từng câu hỏi; mô hình thật phải dùng
  chung 53.259 tham số cho mọi câu, chỉ học 1.000 bước với gradient có nhiễu, và còn phải *khái quát* sang các câu
  thử nó chưa gặp. Đổi lại, nó chỉ tốn một mẫu cho mỗi câu trả lời.
- **Đừng tin công thức $\log N - (N-1)/N$** khi báo cáo đường phần thưởng–KL của best-of-N: ở đây nó đẩy các điểm cam
  sang phải tới khoảng 8 lần, làm best-of-N trông kém học tăng cường. Thật ra mỗi nat KL của nó mua được nhiều độ
  chính xác hơn hẳn: 70,26% ở 0,412 nat, so với 57,64% ở 0,99 nat của học tăng cường với β = 0,3.

## 13 · Thử thay đổi

Các ô dưới chạy lại cùng các hàm với tham số khác. Bạn có thể sửa giá trị rồi chạy lại.

### Một số bất kỳ
Đổi `my_number`. Ô in câu trả lời tham lam của ba mô hình, điểm của hai bộ chấm, và 8 mẫu của SFT-240.

In [ ]:
my_number = "2050"
Xn, Mn = questions([my_number])
print(f"{my_number}: “{' '.join(read(int(my_number)))}”")
for name, p in MODELS.items():
    out = generate(p, Xn, Mn)
    print(
        f"  {NAME[name]:>20}: {text(out)[0]:>13} · bộ kiểm tra {int(verify([my_number], out)[0])}"
        f" · cẩu thả {int(proxy([my_number], out)[0])} · π(đúng) = {prob(math.exp(logprob(p, [my_number], [my_number])[0]))}"
    )
print("  8 mẫu của SFT-240:", text(generate(MODELS["sft"], Xn, Mn, np.random.default_rng(0).random((TMAX, 8)), rep=8)))

### π* trên một câu hỏi khác
`posttrain-cands.json` có 12 câu hỏi (in ra dưới đây). Đổi `which` và `beta_try` để xem $\pi^*$ dồn xác suất thế nào
với từng phần thưởng.

In [ ]:
print("các câu hỏi:", " · ".join(c["digits"] for c in cands["questions"]))
which, beta_try = "554000", 0.1
cq = next(c for c in cands["questions"] if c["digits"] == which)
lab_q, pref_q, r_q, px_q = items(cq)
for reward, rv in (("bộ kiểm tra", r_q), ("bộ cẩu thả", px_q)):
    q, _, kl = star(pref_q, rv, beta_try)
    top3 = " · ".join(f"{lab_q[i]} {pct(q[i])}" for i in np.argsort(-q)[:3])
    print(f"π* theo {reward}, β = {vn(beta_try, 2)}: {top3} · đúng thật {pct(q @ r_q)} · KL {vn(kl, 2)} nat")
print("π_ref:", " · ".join(f"{lab_q[i]} {pct(pref_q[i])}" for i in np.argsort(-pref_q)[:3]))

Với 554000, $\pi_{\text{ref}}$ ưa câu sai “5540000” (thừa một chữ số 0) gấp gần 7 lần câu đúng, nên tối ưu theo bộ cẩu
thả gần như chỉ giữ câu sai.

### Học tăng cường theo bộ chấm cẩu thả
20 bước học theo `proxy` với β = 0, từ SFT-240 (seed 1). Đổi `reward`, `beta_try` hay `n_steps` rồi chạy lại; với
`n_steps` lớn (vài trăm), điểm của bộ cẩu thả tăng rõ trong khi tỉ lệ đọc đúng thật thì không.

In [ ]:
reward, beta_try, n_steps = proxy, 0.0, 20
policy2 = {k: v.copy() for k, v in ref.items()}
opt2, rng2 = Adam(policy2), np.random.default_rng(50_000 + 1)
for _ in range(n_steps):
    rl_step(policy2, ref, opt2, rng2, beta=beta_try, reward=reward)
res = evaluate(policy2, ref, VAL, 1)
print(
    f"sau {n_steps} bước: đúng {pct(res['đúng'])} · cẩu thả {pct(res['cẩu thả'])} · KL {vn(res['KL'], 2)} nat"
    f"  (SFT-240: đúng {pct(before['đúng'])} · cẩu thả {pct(before['cẩu thả'])})"
)

20 bước là quá ít để thấy gì rõ; KL đã lên vài nat vì không có phạt KL (β = 0) và η đủ lớn để mỗi bước đổi mô hình khá
nhiều.

## 14 · Giới hạn và những gì bài này chưa làm

- **Phần thưởng từ người.** Ở đây bộ kiểm tra là một chương trình. Với chatbot thật, “câu trả lời tốt” không có định
  nghĩa chính xác; RLHF học một **mô hình phần thưởng** $r(x, y)$ từ các lựa chọn của người, bằng chính mất mát
  $-\log \sigma(r(y_w) - r(y_l))$ của DPO ở mục 9 (hồi quy logistic của Bài 04 trên hiệu hai điểm). Mô hình phần thưởng
  đó là một thước đo gần đúng, nên mục 10 là chuyện thường ngày chứ không phải chuyện hiếm.
- **PPO và GRPO đầy đủ.** Ta cập nhật một lần cho mỗi lô mẫu. PPO và GRPO dùng lại mỗi lô cho vài bước, nên cần cắt tỉ
  số $\pi_\theta/\pi_{\text{old}}$ để không đi quá xa; PPO còn học một mạng giá trị để có lợi thế cho từng token.
- **Suy luận dài.** DeepSeek-R1-Zero học bằng GRPO chỉ với thưởng đúng–sai và thưởng định dạng `<think>…</think>`, và tự
  viết chuỗi suy luận ngày càng dài; điểm AIME 2024 tăng từ 15,6% lên 71,0% (bản đầu của bài báo, 2025). Mô hình đọc số của ta trả lời trong một
  lượt, không có chỗ để “nghĩ”. DeepSeek-R1 bản đầy đủ còn bắt đầu bằng SFT trên vài nghìn chuỗi suy luận mẫu.
- **Học tăng cường có dạy điều mới không?** Ở đây nó chủ yếu mài sắc: tăng pass@1, giảm pass@k ở $k$ rất lớn (mục 11),
  như Yue và cộng sự (2025). Các nghiên cứu sau (ví dụ ProRL, Liu và cộng sự, 2025) cho thấy học tăng cường đủ lâu, với
  phạt KL và đặt lại $\pi_{\text{ref}}$ định kỳ, có thể giải được cả những câu mà mô hình nền không bao giờ giải được.
  Câu hỏi vẫn đang được tranh luận.

Đây là bài cuối của khóa: từ một nơ-ron đếm phép nhân–cộng (Bài 01) đến một mô hình được dạy trả lời bằng phản hồi.

## Tóm tắt
- Mô hình nền chỉ viết tiếp: GPT-2 small cho “ Paris” xác suất 0,47% sau câu hỏi trần, 67,35% sau hai ví dụ hỏi–đáp.
  Kiến thức có sẵn; SFT dạy thói quen trả lời.
- Học tăng cường không cần nhãn, chỉ cần bộ chấm: SFT-240 (53,25–56,00%) lên 75,33–78,17% với bộ kiểm tra chính xác,
  lệch 3,57–3,99 nat khỏi bản SFT ở β = 0; β = 0,1 cho 67,58–73,25% ở 1,19–1,96 nat. Chỉ 14,9–17,2% nhóm 8 câu trả
  lời (β = 0) có cả đúng lẫn sai; các nhóm còn lại không dạy gì.
- Gradient chính sách: $\nabla J = \mathbb{E}[R\, \nabla \log \pi_\theta(y)]$ với $R = r - \beta \log(\pi_\theta/
  \pi_{\text{ref}})$, đường cơ sở là trung bình 7 câu còn lại; trên bảng nhỏ nó không chệch và hội tụ về
  $\pi^* \propto \pi_{\text{ref}}\, e^{r/\beta}$.
- DPO học thẳng từ các cặp: với 3.000 cặp, phần thưởng ngầm $\beta \log(\pi_\theta/\pi_{\text{ref}})$ khớp $r$ và
  chính sách gần $\pi^*$.
- Goodhart: học theo bộ chấm cẩu thả, điểm của nó lên 88,00–92,67% còn số đọc đúng thật chỉ 44,75–59,67%, dưới cả SFT ở
  2/3 seed. Phạt KL giảm thiệt hại nhưng không sửa được thước đo sai.
- Tính toán lúc trả lời: bỏ phiếu 51,40% → 56,00%, có bộ kiểm tra 70,08% ở 64 mẫu; best-of-64 chỉ lệch 0,412 nat,
  không phải 3,175 nat như công thức $\log N - (N-1)/N$. Học tăng cường thắng ở 1 mẫu, SFT nhỉnh hơn ở $2^{20}$ mẫu.

## Câu hỏi ôn tập
1. Vì sao hai ví dụ hỏi–đáp làm xác suất của “ Paris” tăng từ 0,47% lên 67,35% mà không đổi một trọng số nào? Điều đó
   nói gì về việc SFT dạy gì?
2. Viết gradient của $J = \mathbb{E}_{\pi_\theta}[r] - \beta D_{\mathrm{KL}}(\pi_\theta \| \pi_{\text{ref}})$ dưới dạng
   REINFORCE. Vì sao trừ một đường cơ sở (trung bình 7 câu còn lại) không làm gradient chệch?
3. Trong một nhóm 8 câu trả lời đều sai, lợi thế $A_i$ bằng bao nhiêu khi β = 0? Vì sao điều đó làm câu khó không
   dạy được gì?
4. Chứng minh $\pi^* \propto \pi_{\text{ref}}\, e^{r/\beta}$ là lời giải tối ưu. Nếu $\pi_{\text{ref}}(y) = 0$ thì
   $\pi^*(y)$ bằng bao nhiêu? Điều đó liên quan gì tới câu hỏi “học tăng cường có dạy điều mới không”?
5. Với “năm nghìn hai trăm”, vì sao tối ưu hoàn hảo theo bộ chấm cẩu thả chỉ đúng 29,46%, dù câu đúng luôn được bộ
   chấm cẩu thả nhận?
6. Best-of-64 với SFT-240 lệch bao nhiêu nat khỏi $\pi_{\text{ref}}$, và công thức $\log N - (N-1)/N$ cho bao nhiêu?
   Vì sao công thức sai nhiều đến vậy ở bài này?

<details><summary>Gợi ý đáp án</summary>

1. Hai ví dụ chỉ đổi *ngữ cảnh*: chúng làm câu lệnh giống một trang hỏi–đáp, nơi câu tiếp theo thường là câu trả lời.
   Kiến thức “Paris” đã có từ tiền huấn luyện; SFT dạy hành vi (trả lời rồi dừng), không phải kiến thức mới.
2. $\nabla J = \mathbb{E}[R(y)\, \nabla \log \pi_\theta(y)]$, $R = r - \beta \log(\pi_\theta/\pi_{\text{ref}})$.
   Đường cơ sở $b$ không phụ thuộc $y_i$ nên $\mathbb{E}[b\, \nabla \log \pi_\theta(y_i)] = b\, \nabla \sum_y
   \pi_\theta(y) = b\, \nabla 1 = 0$.
3. $r_i = 0$ với mọi $i$, nên $A_i = 0 - 0 = 0$: gradient bằng 0. Nhóm toàn đúng cũng vậy. Mục 7 thấy chỉ khoảng một
   phần tư số nhóm có tín hiệu.
4. $J(\pi) = -\beta D_{\mathrm{KL}}(\pi \| \pi^*) + \beta \log Z$, lớn nhất khi $\pi = \pi^*$. $\pi^*(y) = 0$ khi
   $\pi_{\text{ref}}(y) = 0$: tối ưu có phạt KL chỉ phân bổ lại xác suất giữa các câu mà mô hình gốc đã có thể viết.
5. $\pi^*$ theo bộ cẩu thả giữ tỉ lệ của $\pi_{\text{ref}}$ giữa các câu bộ cẩu thả nhận; $\pi_{\text{ref}}$ ưa “520”
   (70,00%) hơn “5200” (29,28%), nên câu đúng chỉ giữ phần 29,28 / (70,00 + 29,28 + …) ≈ 29,46%.
6. 0,412 nat so với 3,175 nat. Công thức giả định $N$ mẫu gần như luôn khác nhau; ở đây các câu trả lời trùng nhau rất
   nhiều, và KL thật của từng câu không vượt quá $-\log \pi_{\text{ref}}(\text{đúng})$.
</details>

## Tìm hiểu thêm
- L. Ouyang và cộng sự (2022), [Training language models to follow instructions with human feedback](https://arxiv.org/abs/2203.02155)
  (InstructGPT): SFT trên khoảng 13 nghìn câu lệnh, mô hình phần thưởng, PPO với phạt KL ở từng token ($\beta = 0{,}02$).
- P. Christiano và cộng sự (2017), [Deep RL from Human Preferences](https://arxiv.org/abs/1706.03741); N. Stiennon và
  cộng sự (2020), [Learning to Summarize from Human Feedback](https://arxiv.org/abs/2009.01325).
- C. Zhou và cộng sự (2023), [LIMA: Less Is More for Alignment](https://arxiv.org/abs/2305.11206).
- R. Rafailov và cộng sự (2023), [Direct Preference Optimization](https://arxiv.org/abs/2305.18290).
- L. Gao, J. Schulman, J. Hilton (2022), [Scaling Laws for Reward Model Overoptimization](https://arxiv.org/abs/2210.10760).
- A. Beirami và cộng sự (2024), [Theoretical guarantees on the best-of-n alignment policy](https://arxiv.org/abs/2401.01879):
  $\log N - (N-1)/N$ chỉ là cận trên của KL.
- A. Ahmadian và cộng sự (2024), [Back to Basics: Revisiting REINFORCE Style Optimization for RLHF](https://arxiv.org/abs/2402.14740)
  (RLOO, đường cơ sở trung bình các mẫu còn lại).
- Z. Shao và cộng sự (2024), [DeepSeekMath](https://arxiv.org/abs/2402.03300) (GRPO); DeepSeek-AI (2025),
  [DeepSeek-R1](https://arxiv.org/abs/2501.12948); Q. Yu và cộng sự (2025), [DAPO](https://arxiv.org/abs/2503.14476).
- X. Wang và cộng sự (2022), [Self-Consistency](https://arxiv.org/abs/2203.11171); K. Cobbe và cộng sự (2021),
  [Training Verifiers to Solve Math Word Problems](https://arxiv.org/abs/2110.14168).
- C. Snell và cộng sự (2024), [Scaling LLM Test-Time Compute Optimally](https://arxiv.org/abs/2408.03314).
- Y. Yue và cộng sự (2025), [Does RL Really Incentivize Reasoning Capacity Beyond the Base Model?](https://arxiv.org/abs/2504.13837);
  M. Liu và cộng sự (2025), [ProRL](https://arxiv.org/abs/2505.24864), quan điểm ngược lại.
- N. Lambert, [RLHF Book](https://rlhfbook.com/) (sách mở); L. Weng (2024),
  [Reward Hacking in Reinforcement Learning](https://lilianweng.github.io/posts/2024-11-28-reward-hacking/).

**Bài tiếp theo:** đây là bài cuối của khóa. Hãy quay lại [tổng quan khóa học](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/)
để xem lại cả 26 bài, từ Perceptron đến hậu huấn luyện, và chọn bài bạn muốn làm lại bằng notebook của nó.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")